# Option Surface Data Prep

目标：把某一天的 SPXW 期权链整理成后续 SANOS/XTX 波动率曲面拟合可以使用的标准化输入。

## 第一部分：导入并筛选当天 SPXW 数据

这一部分只做最基础的数据准备：读取 2025 年 SPX index options 文件，筛选 `2025-06-02` 当天、合约代码以 `SPXW` 开头的欧式指数期权，然后按 strike `K` 排序。这里暂时不做价格过滤、不估计 forward，也不做无套利处理。

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.optimize import brentq
from scipy.stats import norm
from sklearn.isotonic import IsotonicRegression
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 30)

DATA_PATH = Path(r"D:\Projects\data\options_data\spx_index_2025.parquet")
YIELD_DIR = Path(r"D:\Projects\data\yield_data")

TRADE_DATE = pd.Timestamp("2025-06-02")
PRODUCT_PREFIX = "SPXW"

DATA_PATH

In [ ]:
columns = [
    "date",
    "ticker",
    "symbol",
    "exdate",
    "cp_flag",
    "strike",
    "strike_price",
    "best_bid",
    "best_offer",
    "mid",
    "bid_ask_spread",
    "volume",
    "open_interest",
    "impl_volatility",
    "delta",
    "gamma",
    "vega",
    "theta",
    "dte",
    "exercise_style",
]

raw = pd.read_parquet(DATA_PATH, columns=columns)
raw.shape

In [ ]:
option_chain = (
    raw.loc[
        raw["date"].eq(TRADE_DATE)
        & raw["symbol"].astype("string").str.startswith(PRODUCT_PREFIX, na=False)
    ]
    .sort_values(["strike", "exdate", "cp_flag", "symbol"], kind="mergesort")
    .reset_index(drop=True)
)

option_chain.shape

In [ ]:
data_summary = {
    "trade_date": option_chain["date"].iloc[0] if len(option_chain) else None,
    "product_prefix": PRODUCT_PREFIX,
    "rows": len(option_chain),
    "tickers": sorted(option_chain["ticker"].dropna().unique().tolist()),
    "exercise_styles": sorted(option_chain["exercise_style"].dropna().unique().tolist()),
    "min_strike": option_chain["strike"].min(),
    "max_strike": option_chain["strike"].max(),
    "expiries": option_chain["exdate"].nunique(),
}

data_summary

In [ ]:
option_chain.head(20)

## 第二部分：讨论并构建贴现因子 DF 和远期 F

SANOS/XTX 论文不是直接在现金 strike 和美元期权价格上拟合，而是先把市场价格转换成 normalized pure option prices。这个转换需要每个到期日的贴现因子 `DF(T)` 和远期 `F(T)`：

`k = K / F(T)`

`pure_price = market_price / (DF(T) * F(T))`

这里先明确本 notebook 的主方法：

`DF(T)` 使用外部 Treasury yield curve 插值得到，因为贴现曲线本质上应当随期限平滑变化，不应该被短端期权报价噪音带着抖动。

`F(T)` 使用 SPXW 期权链内生的 put-call parity implied forward，因为 forward 包含指数股息、融资和市场实际定价信息，适合从同 expiry 的 call/put 报价中估计。

因此主线是：

`Treasury DF + ATM-filtered parity weighted-median F`

其他可选方法只作为诊断：

1. 用外部利率曲线构建 `DF(T)`，再用 put-call parity 估计 `F(T)`。这是当前 notebook 的 baseline，优点是简单、稳定，缺点是 Treasury curve 不是最严格的期权贴现曲线。
2. 对同一 expiry 的 `C - P = a - bK` 做稳健回归，同时估计 `DF=b` 和 `F=a/b`。这更市场内生，但前提是报价质量足够好，并且要认真处理 bid/ask、深虚值、不同步报价等 parity 噪音。
3. 如果有 vendor forward 或 OIS/SOFR discount curve，可以把它作为主输入；parity 回归则作为诊断工具。

下面采用选择 1 作为主线。parity regression 和 robust regression 会保留在图里，但它们只用于检查 parity 数据是否干净，不参与最终 normalized price 的构建。

### 方法论：为什么要转成 normalized 坐标

SANOS/XTX 论文的核心不是直接拟合现金 strike `K` 和美元价格 `C_market(T, K)`，而是先把每个到期日都转换到一个统一的 forward-normalized 坐标里。

对每个到期日 `T`，把指数终值写成：

`S_T = F_T * Z_T`

其中 `F_T` 是今天市场隐含的到期 forward，`Z_T = S_T / F_T` 是标准化后的资产。这样定义以后，`Z_T` 的均值是 1：

`E[Z_T] = 1`

直觉上，`F_T` 负责描述这个 expiry 的指数水平，比如 SPX 大约在 5950、6000 或 6100；`Z_T` 只描述相对于 forward 的随机波动，比如未来是 forward 的 0.95、1.00 或 1.05 倍。

strike 也必须用同一个单位标准化。原始 payoff 是：

`(S_T - K)+`

代入 `S_T = F_T * Z_T`，并定义 `k = K / F_T`，就得到：

`(F_T * Z_T - K)+ = (F_T * Z_T - F_T * k)+ = F_T * (Z_T - k)+`

所以市场价格可以写成：

`C_market(T, K) = DF_T * F_T * E[(Z_T - k)+]`

因此论文里真正拟合的是：

`C_pure(T, k) = C_market(T, K) / (DF_T * F_T) = E[(Z_T - k)+]`

这一步的意义是把所有 expiry 都放进同一个坐标系：ATM 附近都是 `k ≈ 1`，OTM call 是 `k > 1`，OTM put 是 `k < 1`。后续 SANOS 的离散密度权重 `q` 也因此有统一约束：

`sum(q_i) = 1`

`sum(q_i * k_i) = 1`

如果不做这个标准化，不同 expiry 的 forward 水平、贴现尺度和 strike 单位会混在一起，后面的无套利拟合就很难写成论文里的 martingale distribution 问题。

In [ ]:
yield_files = {
    "DGS1MO": 1 / 12,
    "DGS3MO": 3 / 12,
    "DGS6MO": 6 / 12,
    "DGS1": 1.0,
    "DGS2": 2.0,
}

curve_points = []
for series, tenor_years in yield_files.items():
    rates = pd.read_csv(YIELD_DIR / f"{series}.csv", parse_dates=["observation_date"])
    row = rates.loc[rates["observation_date"].eq(TRADE_DATE), series]
    if not row.empty and pd.notna(row.iloc[0]):
        curve_points.append((tenor_years, row.iloc[0] / 100.0))

yield_curve = pd.DataFrame(curve_points, columns=["tenor_years", "rate"])
yield_curve

### 用 Treasury yield curve 构建主贴现因子 DF

这里用 FRED Treasury constant maturity yields 作为外部利率输入。对每个 expiry，先计算 `T = dte / 365`，再在 `1M, 3M, 6M, 1Y, 2Y` 这些期限上对利率做线性插值，最后得到：

`DF(T) = exp(-r(T) * T)`

这个 `DF` 是后续 normalized price 的主贴现因子。我们不使用 parity regression 得到的 `DF` 作为主输入，因为短期限时回归斜率容易受到 bid/ask、报价不同步和微结构噪音影响，图上也能看到 regression DF 在短端更容易抖动。

In [ ]:
def interpolated_rate(years: pd.Series) -> np.ndarray:
    xp = yield_curve["tenor_years"].to_numpy()
    fp = yield_curve["rate"].to_numpy()
    return np.interp(years.to_numpy(dtype=float), xp, fp)


expiry_terms = (
    option_chain[["exdate", "dte"]]
    .drop_duplicates()
    .sort_values("exdate")
    .reset_index(drop=True)
)
expiry_terms["T"] = expiry_terms["dte"] / 365.0
expiry_terms["rate"] = interpolated_rate(expiry_terms["T"])
expiry_terms["df"] = np.exp(-expiry_terms["rate"] * expiry_terms["T"])

expiry_terms.head(10)

### 用 put-call parity 估计每个到期日的 forward

对同一个 expiry 和 strike，欧式期权理论上满足 `C - P = DF * (F - K)`。因此每个 paired strike 都可以给出一个 implied forward：

`F_K = K + (C - P) / DF`

现实数据会有 bid/ask、深虚值报价、不同步报价等噪音，所以这里不取所有点的简单平均。主线 forward 构建规则如下：

1. 只使用同一 expiry、同一 strike 下 call 和 put 都存在的 paired quotes。
2. 过滤掉 bid/ask 不合法、spread 非正、关键字段缺失的点。
3. 用 Treasury `DF(T)` 计算每个 strike 的 `F_K = K + (C_mid - P_mid) / DF(T)`。
4. 用所有合法 `F_K` 的中位数粗估 `F0`。
5. 只保留 ATM 附近的点，默认 `abs(K / F0 - 1) <= 5%`。
6. 在 ATM 样本内继续过滤 pair spread 较宽的点，默认保留 spread 较窄的 60%。
7. 对剩余 `F_K` 使用 `1 / pair_spread` 加权中位数，得到最终主 forward `F(T)`。

这样做的直觉是：`DF` 交给平滑的外部利率曲线，`F` 交给期权市场自身，但只让 ATM 附近、报价质量较好的 parity 点参与估计。

In [ ]:
paired_mid = option_chain.pivot_table(
    index=["exdate", "dte", "strike"],
    columns="cp_flag",
    values=["mid", "best_bid", "best_offer", "bid_ask_spread"],
    aggfunc="mean",
)
paired_mid.columns = [f"{flag}_{field}" for field, flag in paired_mid.columns]
paired_mid = paired_mid.reset_index()
paired_mid = paired_mid.merge(expiry_terms[["exdate", "T", "rate", "df"]], on="exdate", how="left")
paired_mid = paired_mid.dropna(subset=["C_mid", "P_mid", "C_best_bid", "P_best_bid", "C_best_offer", "P_best_offer", "C_bid_ask_spread", "P_bid_ask_spread", "df"])
paired_mid = paired_mid.loc[
    paired_mid["C_best_offer"].gt(paired_mid["C_best_bid"])
    & paired_mid["P_best_offer"].gt(paired_mid["P_best_bid"])
    & paired_mid["C_bid_ask_spread"].gt(0)
    & paired_mid["P_bid_ask_spread"].gt(0)
].copy()
paired_mid["pair_spread"] = paired_mid["C_bid_ask_spread"] + paired_mid["P_bid_ask_spread"]
paired_mid["implied_forward_at_strike"] = paired_mid["strike"] + (paired_mid["C_mid"] - paired_mid["P_mid"]) / paired_mid["df"]

def weighted_median(values: pd.Series, weights: pd.Series) -> float:
    order = np.argsort(values.to_numpy(dtype=float))
    sorted_values = values.to_numpy(dtype=float)[order]
    sorted_weights = weights.to_numpy(dtype=float)[order]
    cutoff = 0.5 * sorted_weights.sum()
    return sorted_values[np.searchsorted(np.cumsum(sorted_weights), cutoff)]


def select_forward_quotes(expiry_pairs: pd.DataFrame, atm_width: float = 0.05, spread_quantile: float = 0.60, min_points: int = 8) -> pd.DataFrame:
    clean = expiry_pairs.replace([np.inf, -np.inf], np.nan).dropna(subset=["strike", "pair_spread", "implied_forward_at_strike"])
    clean = clean.loc[clean["pair_spread"].gt(0)].copy()
    if len(clean) < min_points:
        return clean

    f0 = clean["implied_forward_at_strike"].median()
    atm = clean.loc[(clean["strike"] / f0 - 1.0).abs().le(atm_width)].copy()
    if len(atm) < min_points:
        atm = clean.assign(distance=(clean["strike"] - f0).abs()).nsmallest(min_points, "distance").copy()

    spread_cutoff = atm["pair_spread"].quantile(spread_quantile)
    filtered = atm.loc[atm["pair_spread"].le(spread_cutoff)].copy()
    if len(filtered) < min_points:
        filtered = atm.copy()
    return filtered


def atm_filtered_parity_forward(expiry_pairs: pd.DataFrame) -> pd.Series:
    filtered = select_forward_quotes(expiry_pairs)
    if filtered.empty:
        return pd.Series({"forward": np.nan, "forward_points": 0})

    weights = 1.0 / filtered["pair_spread"]
    forward = weighted_median(filtered["implied_forward_at_strike"], weights)
    return pd.Series({"forward": forward, "forward_points": len(filtered)})


forwards = (
    paired_mid.groupby("exdate")[["strike", "pair_spread", "implied_forward_at_strike"]]
    .apply(atm_filtered_parity_forward)
    .reset_index()
)

expiry_terms = expiry_terms.merge(forwards, on="exdate", how="left")
expiry_terms.head(10)

### 诊断：parity regression 与 robust regression

这一小节不是主方法，而是诊断工具。主方法已经固定为 `Treasury DF + ATM-filtered parity weighted-median F`。

为了检查 put-call parity 数据是否干净，我们额外在同一批 ATM + spread-filtered quotes 上做 parity regression：

`C_mid - P_mid = a - bK`

如果市场报价完全同步且无噪音，那么同一 expiry 下这应当是一条直线，其中 `b = DF`，`a = DF * F`，所以 `F = a / b`。

但是短期限时，`DF` 非常接近 1，回归斜率会被很小的 mid 噪音、bid/ask 和报价不同步放大。因此 regression 得到的 `DF` 只用于诊断，不用于最终标准化。

这里同时做两个诊断版本：

1. 加权最小二乘：复用主 forward 的 ATM + spread filter 样本，权重取 `1 / (call_spread + put_spread)`。
2. Huber-style robust 回归：在加权最小二乘基础上按残差迭代降权，减少异常 strike 对斜率 `DF` 的影响。

图里如果 regression forward 与主 forward 接近，说明 parity 对 forward 的估计比较干净；如果 regression DF 在短端抖动，则说明它不适合作为主贴现曲线，这也是我们选择 Treasury DF 的原因。

In [ ]:
paired_quotes = paired_mid.copy()
paired_quotes["parity_y"] = paired_quotes["C_mid"] - paired_quotes["P_mid"]

def fit_weighted_line(x: np.ndarray, y: np.ndarray, w: np.ndarray) -> tuple[float, float]:
    X = np.column_stack([np.ones_like(x), x])
    Xw = X * np.sqrt(w)[:, None]
    yw = y * np.sqrt(w)
    return tuple(np.linalg.lstsq(Xw, yw, rcond=None)[0])


def fit_huber_line(x: np.ndarray, y: np.ndarray, base_w: np.ndarray, max_iter: int = 8, c: float = 1.345) -> tuple[float, float]:
    robust_w = base_w.copy()
    intercept, slope = fit_weighted_line(x, y, robust_w)
    for _ in range(max_iter):
        residual = y - (intercept + slope * x)
        scale = 1.4826 * np.median(np.abs(residual - np.median(residual)))
        if not np.isfinite(scale) or scale <= 1e-12:
            break
        z = np.abs(residual) / scale
        huber_w = np.where(z <= c, 1.0, c / z)
        new_intercept, new_slope = fit_weighted_line(x, y, base_w * huber_w)
        if np.hypot(new_intercept - intercept, new_slope - slope) < 1e-10:
            intercept, slope = new_intercept, new_slope
            break
        intercept, slope = new_intercept, new_slope
    return intercept, slope


def weighted_parity_regression(expiry_pairs: pd.DataFrame) -> pd.Series:
    clean = select_forward_quotes(expiry_pairs)
    clean = clean.replace([np.inf, -np.inf], np.nan).dropna(subset=["strike", "parity_y", "pair_spread"])
    if len(clean) < 5:
        return pd.Series({"df_reg": np.nan, "forward_reg": np.nan, "df_robust": np.nan, "forward_robust": np.nan, "parity_points": len(clean), "parity_rmse": np.nan, "parity_rmse_robust": np.nan})

    x = clean["strike"].to_numpy(dtype=float)
    y = clean["parity_y"].to_numpy(dtype=float)
    w = 1.0 / clean["pair_spread"].to_numpy(dtype=float)
    intercept, slope = fit_weighted_line(x, y, w)
    intercept_robust, slope_robust = fit_huber_line(x, y, w)

    df_reg = -slope
    forward_reg = intercept / df_reg if df_reg > 0 else np.nan
    df_robust = -slope_robust
    forward_robust = intercept_robust / df_robust if df_robust > 0 else np.nan
    fitted = intercept + slope * x
    fitted_robust = intercept_robust + slope_robust * x
    rmse = np.sqrt(np.average((y - fitted) ** 2, weights=w))
    rmse_robust = np.sqrt(np.average((y - fitted_robust) ** 2, weights=w))
    return pd.Series({"df_reg": df_reg, "forward_reg": forward_reg, "df_robust": df_robust, "forward_robust": forward_robust, "parity_points": len(clean), "parity_rmse": rmse, "parity_rmse_robust": rmse_robust})

parity_regression = (
    paired_quotes.groupby("exdate")[["strike", "parity_y", "pair_spread", "implied_forward_at_strike"]]
    .apply(weighted_parity_regression)
    .reset_index()
)

method_compare = expiry_terms.merge(parity_regression, on="exdate", how="left")
method_compare["forward_diff"] = method_compare["forward_reg"] - method_compare["forward"]
method_compare["df_diff"] = method_compare["df_reg"] - method_compare["df"]
method_compare["forward_robust_diff"] = method_compare["forward_robust"] - method_compare["forward"]
method_compare["df_robust_diff"] = method_compare["df_robust"] - method_compare["df"]
method_compare.head(10)

### 诊断图的读法

上图比较 forward：主 forward 是 `Treasury DF + ATM parity weighted median F`，regression/robust forward 只是检查 parity 线性关系是否稳定。如果三条 forward 很接近，说明用 parity 估计 `F(T)` 是可靠的。

下图比较 DF：绿色 Treasury DF 是主贴现曲线；parity regression DF 和 robust DF 只是诊断线。如果短端 regression DF 有抖动，不代表真实贴现曲线也抖动，而更可能是短期期权报价噪音被回归斜率放大。

In [ ]:
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=("Forward F(T)", "Discount factor DF(T)"),
)

fig.add_trace(
    go.Scatter(
        x=method_compare["dte"],
        y=method_compare["forward"],
        mode="lines+markers",
        name="Treasury DF + parity median F",
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=method_compare["dte"],
        y=method_compare["forward_reg"],
        mode="lines+markers",
        name="Parity regression F",
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=method_compare["dte"],
        y=method_compare["forward_robust"],
        mode="lines+markers",
        name="Parity robust F",
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=method_compare["dte"],
        y=method_compare["df"],
        mode="lines+markers",
        name="Treasury interpolated DF",
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=method_compare["dte"],
        y=method_compare["df_reg"],
        mode="lines+markers",
        name="Parity regression DF",
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=method_compare["dte"],
        y=method_compare["df_robust"],
        mode="lines+markers",
        name="Parity robust DF",
    ),
    row=2,
    col=1,
)

fig.update_xaxes(title_text="DTE", row=2, col=1)
fig.update_yaxes(title_text="Forward", row=1, col=1)
fig.update_yaxes(title_text="DF", row=2, col=1)
fig.update_layout(height=720, title=f"{PRODUCT_PREFIX} {TRADE_DATE.date()} forward / discount construction comparison")
fig.show()

### 转换成论文使用的 normalized 坐标

这里进入 SANOS/XTX 论文真正需要的坐标变换。实际使用的主输入是：

`df = Treasury interpolated DF`

`forward = ATM-filtered parity weighted-median F`

然后把现金 strike 和美元价格转换成 normalized strike `k` 与 pure option prices：

`k = strike / forward`

`pure_bid = best_bid / (df * forward)`

`pure_mid = mid / (df * forward)`

`pure_ask = best_offer / (df * forward)`

后续 SANOS/XTX 的线性规划拟合会在这些 normalized prices 上进行。

In [ ]:
option_chain = option_chain.merge(
    expiry_terms[["exdate", "T", "rate", "df", "forward"]],
    on="exdate",
    how="left",
)

option_chain["k"] = option_chain["strike"] / option_chain["forward"]
option_chain["pure_bid"] = option_chain["best_bid"] / (option_chain["df"] * option_chain["forward"])
option_chain["pure_mid"] = option_chain["mid"] / (option_chain["df"] * option_chain["forward"])
option_chain["pure_ask"] = option_chain["best_offer"] / (option_chain["df"] * option_chain["forward"])

option_chain = option_chain.sort_values(["k", "exdate", "cp_flag", "symbol"], kind="mergesort").reset_index(drop=True)
option_chain.head(20)

## 第三部分：构造 SANOS 拟合输入

完成 normalized 坐标之后，下一步不是直接看每个点的 implied vol，而是把市场报价整理成 SANOS/XTX 线性规划可以使用的 call price surface 输入。

论文的 production model 拟合的是 normalized call price：

`C_hat(T_j, k) = sum_i q_j^i * Call(k_i, k, eta * V_j)`

因此这一部分先做三件事：

1. 只取 call options，保留 `pure_bid / pure_mid / pure_ask`。put 已经用于估计 forward，后续主拟合用 call surface。
2. 做基础报价过滤，去掉 bid/ask 不合法、normalized price 缺失或非有限的点。
3. 按 expiry 统计 normalized strike 覆盖范围、报价点数量，并为每个 expiry 构造一个初步 ATM total variance `V_j`。

这里还不求解 `q_j`，也不构造完整 Black-Scholes kernel。目标只是得到干净的 SANOS 输入表。

In [ ]:
call_surface = option_chain.loc[
    option_chain["cp_flag"].eq("C")
    & option_chain["best_offer"].gt(option_chain["best_bid"])
    & option_chain["best_bid"].ge(0)
    & option_chain["pure_bid"].ge(0)
    & option_chain["pure_ask"].gt(option_chain["pure_bid"])
    & np.isfinite(option_chain["k"])
    & np.isfinite(option_chain["pure_mid"])
].copy()

call_surface = call_surface.sort_values(["exdate", "k"], kind="mergesort").reset_index(drop=True)
call_surface.shape

### 每个 expiry 的覆盖范围诊断

SANOS 后面会在每个 expiry 上拟合一条 normalized call curve。这里先检查每个 expiry 有多少 call 报价、`k` 覆盖范围如何、bid/ask spread 大概多宽。

如果某个 expiry 的点数太少，或者 `k` 覆盖范围太窄，后续拟合出来的密度会不稳定。

In [ ]:
expiry_quote_summary = (
    call_surface.assign(pure_spread=call_surface["pure_ask"] - call_surface["pure_bid"])
    .groupby("exdate")
    .agg(
        dte=("dte", "first"),
        T=("T", "first"),
        forward=("forward", "first"),
        df=("df", "first"),
        n_calls=("k", "size"),
        min_k=("k", "min"),
        max_k=("k", "max"),
        median_k=("k", "median"),
        median_spread=("bid_ask_spread", "median"),
        median_pure_spread=("pure_spread", "median"),
    )
    .reset_index()
)

expiry_quote_summary.head(15)

### 初步构造 backbone total variance `V_j`

论文 production model 需要每个 expiry 的一个 backbone variance `V_j`，并要求它随 maturity 递增。这里不直接使用数据字段里的 `impl_volatility`，而是把 normalized call price 代回 Black-Scholes 模型自己反解 implied volatility：

`V_j = sigma_atm(T_j)^2 * T_j`

这里的 `sigma_atm` 取每个 expiry 中离 `k=1` 最近、且能从 `pure_mid` 成功反解 Black-Scholes IV 的 call。数据字段里的 `impl_volatility` 只保留为对照，不参与 `V_j` 构造。

注意：这里先验证 raw ATM total variance 是否随 maturity 非递减。如果没有违例，就直接使用 raw 值；如果存在 calendar 方向的 total variance 违例，则用 isotonic regression 找一条尽量贴近 raw 值、但强制非递减的 `V_j` 曲线。

In [ ]:
def bs_call_forward_unit(k, total_variance):
    total_variance = max(float(total_variance), 0.0)
    if total_variance < 1e-14:
        return max(1.0 - float(k), 0.0)

    vol_sqrt_t = np.sqrt(total_variance)
    d1 = (-np.log(float(k)) + 0.5 * total_variance) / vol_sqrt_t
    d2 = d1 - vol_sqrt_t
    return norm.cdf(d1) - float(k) * norm.cdf(d2)


def bs_put_forward_unit(k, total_variance):
    total_variance = max(float(total_variance), 0.0)
    if total_variance < 1e-14:
        return max(float(k) - 1.0, 0.0)

    vol_sqrt_t = np.sqrt(total_variance)
    d1 = (-np.log(float(k)) + 0.5 * total_variance) / vol_sqrt_t
    d2 = d1 - vol_sqrt_t
    return float(k) * norm.cdf(-d2) - norm.cdf(-d1)


def implied_vol_from_pure_option(k: float, pure_price: float, T: float, cp_flag: str) -> float:
    """Invert normalized forward-unit option price into Black-Scholes IV.

    k is K/F, and the price is divided by DF*F. For IV diagnostics we use
    OTM options: puts when k < 1 and calls when k >= 1.
    """
    k = float(k)
    pure_price = float(pure_price)
    T = float(T)
    cp_flag = str(cp_flag).upper()[0]

    if not np.isfinite(k) or not np.isfinite(pure_price) or not np.isfinite(T) or k <= 0 or T <= 0:
        return np.nan

    if cp_flag == "P":
        lower = max(k - 1.0, 0.0)
        upper = k
        pricing_fn = bs_put_forward_unit
    else:
        lower = max(1.0 - k, 0.0)
        upper = 1.0
        pricing_fn = bs_call_forward_unit

    tol = 1e-10
    pure_price = min(max(pure_price, lower + tol), upper - tol)

    def objective(sigma):
        return pricing_fn(k, sigma * sigma * T) - pure_price

    try:
        return brentq(objective, 1e-6, 6.0, maxiter=100)
    except ValueError:
        return np.nan


def implied_vol_from_pure_call(k: float, pure_price: float, T: float) -> float:
    """Backward-compatible helper for ATM-call total-variance construction."""
    return implied_vol_from_pure_option(k, pure_price, T, "C")


def otm_cp_flag_from_k(k: float) -> str:
    return "C" if float(k) >= 1.0 else "P"


def otm_price_from_pure_call(k: float, pure_call_price: float) -> float:
    """Convert fitted pure call into the corresponding OTM option price.

    In normalized forward units, put-call parity is C - P = 1 - k,
    so the put price is P = C + k - 1.
    """
    k = float(k)
    pure_call_price = float(pure_call_price)
    if k >= 1.0:
        return pure_call_price
    return pure_call_price + k - 1.0


def bs_vega_forward_unit(k: float, sigma: float, T: float) -> float:
    k = float(k)
    sigma = float(sigma)
    T = float(T)
    if not np.isfinite(k) or not np.isfinite(sigma) or not np.isfinite(T) or k <= 0 or sigma <= 0 or T <= 0:
        return np.nan
    total_variance = sigma * sigma * T
    if total_variance < 1e-14:
        return np.nan
    vol_sqrt_t = np.sqrt(total_variance)
    d1 = (-np.log(k) + 0.5 * total_variance) / vol_sqrt_t
    return norm.pdf(d1) * np.sqrt(T)


call_surface["bs_iv"] = [
    implied_vol_from_pure_call(k, price, T)
    for k, price, T in zip(call_surface["k"], call_surface["pure_mid"], call_surface["T"])
]
call_surface["bs_total_variance"] = call_surface["bs_iv"] ** 2 * call_surface["T"]

call_surface[["exdate", "dte", "strike_price", "k", "pure_mid", "bs_iv", "bs_total_variance"]].head()

atm_candidates = call_surface[
    np.isfinite(call_surface["bs_iv"])
    & call_surface["bs_iv"].gt(0)
    & np.isfinite(call_surface["bs_total_variance"])
    & call_surface["bs_total_variance"].gt(0)
].copy()
atm_candidates["abs_moneyness"] = (atm_candidates["k"] - 1.0).abs()

atm_idx = atm_candidates.groupby("exdate")["abs_moneyness"].idxmin()
atm_rows = (
    atm_candidates.loc[
        atm_idx,
        ["exdate", "dte", "T", "k", "bs_iv", "bs_total_variance"]
    ]
    .rename(
        columns={
            "k": "atm_k",
            "bs_iv": "atm_iv",
            "bs_total_variance": "atm_total_variance_raw",
        }
    )
    .sort_values("T", kind="mergesort")
    .reset_index(drop=True)
)

atm_rows["variance_diff_raw"] = atm_rows["atm_total_variance_raw"].diff()
atm_rows["variance_monotone_violation"] = atm_rows["variance_diff_raw"].lt(-1e-12).fillna(False)
has_variance_violation = bool(atm_rows["variance_monotone_violation"].any())

if has_variance_violation:
    iso = IsotonicRegression(increasing=True, out_of_bounds="clip")
    atm_rows["atm_total_variance"] = iso.fit_transform(
        atm_rows["T"].to_numpy(),
        atm_rows["atm_total_variance_raw"].to_numpy(),
    )
else:
    atm_rows["atm_total_variance"] = atm_rows["atm_total_variance_raw"]

atm_rows["atm_total_variance_adjustment"] = (
    atm_rows["atm_total_variance"] - atm_rows["atm_total_variance_raw"]
)

atm_rows.head(15)

### Total variance 单调性诊断

这里检查 `atm_total_variance_raw` 是否随 `T` 非递减。如果存在违例，`atm_total_variance` 是 isotonic regression 后的修正值；如果不存在违例，二者完全相同。

In [ ]:
variance_monotonicity_summary = {
    "has_variance_violation": has_variance_violation,
    "violation_count": int(atm_rows["variance_monotone_violation"].sum()),
    "max_abs_adjustment": atm_rows["atm_total_variance_adjustment"].abs().max(),
}

variance_monotonicity_summary

### SANOS 输入表

下面把市场报价整理成 LP 使用的 normalized call-equivalent quotes。

这里不再只用原始 call quotes。原因是交易上观察 volatility smile 时，左翼通常看 OTM put，右翼看 OTM call；如果 LP 只拟合 call，而诊断图左翼拿 market put IV 对比，就会把 put/call parity 噪音也混进“模型误差”里，尤其 1D 短端会非常明显。

因此这里使用 OTM 口径：

- `k >= 1`：直接使用 call quote；
- `k < 1`：使用 put quote，并通过 normalized parity 转成 call-equivalent price；
- normalized parity 为 `C - P = 1 - k`，所以 `C = P + 1 - k`；
- bid/ask 同样平移：`call_equiv_bid = put_bid + 1 - k`，`call_equiv_ask = put_ask + 1 - k`。

这样 LP 仍然拟合的是 pure call price surface，但输入价格来自 OTM 市场报价，和后面的 OTM IV 诊断口径一致。

In [ ]:
FIT_MIN_VEGA_OVER_SQRT_T = 1e-3
FIT_MIN_QUOTES_PER_EXPIRY = 20
FIT_MIN_VOLUME = 30        # moderate liquidity filter: >=30 lots traded that day
FIT_MIN_NORMALIZED_K = 0.45  # moneyness floor: removes the deep wing that breaks the
                             # LP numerically (liquidity alone can't -- volume>=30 with
                             # no k-cut leaves min_k~0.39 and the LP fails Status 4).
                             # The two filters are COMPLEMENTARY: k-cut is surgical on
                             # the strike axis (clean deep wing, all maturities), volume
                             # is on the liquidity axis (drops sparse/long-dated junk).

sanos_market_quotes = option_chain.loc[
    option_chain["T"].gt(0)
    & option_chain["k"].ge(FIT_MIN_NORMALIZED_K)
    & option_chain["volume"].ge(FIT_MIN_VOLUME)
    & (
        (option_chain["cp_flag"].eq("C") & option_chain["k"].ge(1.0))
        | (option_chain["cp_flag"].eq("P") & option_chain["k"].lt(1.0))
    )
    & option_chain["best_offer"].gt(option_chain["best_bid"])
    & option_chain["best_bid"].ge(0)
    & option_chain["pure_bid"].ge(0)
    & option_chain["pure_ask"].gt(option_chain["pure_bid"])
    & np.isfinite(option_chain["k"])
    & np.isfinite(option_chain["pure_mid"])
].copy()

sanos_market_quotes["source_otm_iv"] = [
    implied_vol_from_pure_option(k, price, T, cp)
    for k, price, T, cp in zip(
        sanos_market_quotes["k"],
        sanos_market_quotes["pure_mid"],
        sanos_market_quotes["T"],
        sanos_market_quotes["cp_flag"],
    )
]
sanos_market_quotes["source_vega"] = [
    bs_vega_forward_unit(k, iv, T)
    for k, iv, T in zip(
        sanos_market_quotes["k"],
        sanos_market_quotes["source_otm_iv"],
        sanos_market_quotes["T"],
    )
]
sanos_market_quotes["source_vega_over_sqrt_t"] = sanos_market_quotes["source_vega"] / np.sqrt(sanos_market_quotes["T"])
sanos_market_quotes = sanos_market_quotes.loc[
    np.isfinite(sanos_market_quotes["source_otm_iv"])
    & np.isfinite(sanos_market_quotes["source_vega_over_sqrt_t"])
    & sanos_market_quotes["source_vega_over_sqrt_t"].ge(FIT_MIN_VEGA_OVER_SQRT_T)
].copy()

active_expiries = (
    sanos_market_quotes.groupby("exdate")["k"]
    .size()
    .loc[lambda s: s.ge(FIT_MIN_QUOTES_PER_EXPIRY)]
    .index
)
sanos_market_quotes = sanos_market_quotes.loc[sanos_market_quotes["exdate"].isin(active_expiries)].copy()

put_to_call_shift = np.where(sanos_market_quotes["cp_flag"].eq("P"), 1.0 - sanos_market_quotes["k"], 0.0)
sanos_market_quotes["source_cp_flag"] = sanos_market_quotes["cp_flag"]
sanos_market_quotes["source_pure_bid"] = sanos_market_quotes["pure_bid"]
sanos_market_quotes["source_pure_mid"] = sanos_market_quotes["pure_mid"]
sanos_market_quotes["source_pure_ask"] = sanos_market_quotes["pure_ask"]
sanos_market_quotes["pure_bid"] = sanos_market_quotes["pure_bid"] + put_to_call_shift
sanos_market_quotes["pure_mid"] = sanos_market_quotes["pure_mid"] + put_to_call_shift
sanos_market_quotes["pure_ask"] = sanos_market_quotes["pure_ask"] + put_to_call_shift
sanos_market_quotes["pure_bid"] = sanos_market_quotes["pure_bid"].clip(lower=0.0)
sanos_market_quotes["pure_mid"] = sanos_market_quotes["pure_mid"].clip(lower=0.0)
sanos_market_quotes["pure_ask"] = np.maximum(sanos_market_quotes["pure_ask"], sanos_market_quotes["pure_bid"] + 1e-12)

sanos_quotes = sanos_market_quotes.merge(
    atm_rows[["exdate", "atm_k", "atm_iv", "atm_total_variance_raw", "atm_total_variance", "atm_total_variance_adjustment", "variance_monotone_violation"]],
    on="exdate",
    how="left",
)

sanos_quotes = sanos_quotes[[
    "date", "exdate", "dte", "T", "symbol", "source_cp_flag", "strike", "k",
    "best_bid", "best_offer", "mid",
    "source_pure_bid", "source_pure_mid", "source_pure_ask",
    "pure_bid", "pure_mid", "pure_ask",
    "impl_volatility", "forward", "df",
    "atm_k", "atm_iv", "atm_total_variance_raw", "atm_total_variance", "atm_total_variance_adjustment", "variance_monotone_violation",
    "volume", "open_interest", "bid_ask_spread",
]].sort_values(["exdate", "k"], kind="mergesort").reset_index(drop=True)

sanos_quotes.head(20)

In [ ]:
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=False,
    vertical_spacing=0.10,
    subplot_titles=("Normalized call quotes by expiry", "ATM total variance backbone"),
)

plot_expiries = (
    sanos_quotes.groupby("exdate")["dte"]
    .first()
    .sort_values()
    .head(12)
    .index
)
for exdate in plot_expiries:
    one = sanos_quotes.loc[sanos_quotes["exdate"].eq(exdate)]
    label = f"{pd.Timestamp(exdate).date()} ({int(one['dte'].iloc[0])}D)"
    fig.add_trace(
        go.Scatter(
            x=one["k"],
            y=one["pure_mid"],
            mode="markers",
            name=label,
            marker=dict(size=4),
        ),
        row=1,
        col=1,
    )

fig.add_trace(
    go.Scatter(
        x=atm_rows["dte"],
        y=atm_rows["atm_total_variance_raw"],
        mode="markers+lines",
        name="raw ATM total variance",
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=atm_rows["dte"],
        y=atm_rows["atm_total_variance"],
        mode="markers+lines",
        name="monotone ATM total variance",
    ),
    row=2,
    col=1,
)

fig.update_xaxes(title_text="normalized strike k", row=1, col=1)
fig.update_yaxes(title_text="pure call mid", row=1, col=1)
fig.update_xaxes(title_text="DTE", row=2, col=1)
fig.update_yaxes(title_text="total variance", row=2, col=1)
fig.update_layout(height=850, title=f"{PRODUCT_PREFIX} {TRADE_DATE.date()} SANOS input diagnostics")
fig.show()

## 第四部分：构造 SANOS kernel 输入

### 方法论：用 Black-Scholes call basis 构造平滑无套利曲线

到第三部分为止，我们已经把市场报价转换成 normalized call prices：

`C_pure(T_j, k_l)`

SANOS production model 的下一步，是把每个 expiry 的 call curve 写成很多个 Black-Scholes call basis 的加权和：

`C_hat(T_j, k) = sum_i q_j^i * Call(k_i, k, eta * V_j)`

这里可以从两个角度理解。

**函数逼近角度**

每一个 `Call(k_i, k, eta * V_j)` 都是一条以 anchor `k_i` 为中心的平滑 call 曲线。模型用非负权重 `q_j^i` 把这些 basis calls 加起来，拟合市场上的整条 call price curve。

**风险中性分布角度**

先抽一个离散状态：

`X_j = k_i with probability q_j^i`

再乘上一个均值为 1 的 lognormal shock：

`Y_j = exp(-0.5 * eta * V_j + sqrt(eta * V_j) * N)`

最终：

`Z_j = X_j * Y_j`

于是：

`E[(Z_j - k)+] = sum_i q_j^i * E[(k_i * Y_j - k)+]`

而右边的 `E[(k_i * Y_j - k)+]` 正是 `Call(k_i, k, eta * V_j)`。所以这个模型不是随便拿多个 call 做拟合；它对应一个合法的风险中性分布。只要后续 LP 约束：

`q_j^i >= 0`

`sum_i q_j^i = 1`

`sum_i q_j^i * k_i = 1`

就能保证 normalized underlying 的均值为 1，也就是 martingale 条件。

### 为什么 production model 只用每个 expiry 一个 `V_j`

论文的 production model 使用：

`Call(k_i, k, eta * V_j)`

这里 `V_j` 只依赖 expiry，不依赖 strike。它是每个 expiry 的 ATM total variance backbone，而不是每个期权各自的 implied variance。

这样做有两个原因：

1. **保持线性规划结构简单。** 一旦 `V_j` 固定，所有 kernel matrix `G_j` 都是已知的，后续只需要线性地求 `q_j`。
2. **让 smile/skew 主要由 `q_j` 表达。** `V_j` 只决定每个 basis 的平滑宽度；真正的分布形状、偏度和尾部主要由离散权重 `q_j` 学出来。

论文确实讨论过更一般的版本，例如每个 strike 一个 `V_j^i`，但那会让时间方向无套利和校准都更复杂，不是 production model 主线。

因此我们当前复刻的是 production model：

`V_j = ATM total variance per expiry`

`kernel variance = eta * V_j`

### eta 的意义：额外平滑强度，不是真实总方差

`eta` 控制的是 lognormal smoothing shock 的宽度：

`smooth_variance = eta * V_j`

它不是说“真实市场方差只用 eta 这一部分”。真实 smile/surface 的形状由两部分共同表达：

1. `q_j` 的离散分布；
2. 每个 anchor 上额外乘的 lognormal smoothing shock。

如果 `eta = 0`：

`Call(k_i, k, 0) = (k_i - k)+`

模型退化成离散线性插值。它很容易贴合报价，但曲线不光滑，密度是离散尖点。

如果 `eta > 0`：

每个离散 anchor `k_i` 被一个 Black-Scholes/lognormal kernel 平滑成一个小分布，最终 call curve 和密度都会更光滑。

如果 `eta = 1`：

每个 basis 都被完整 ATM total variance 平滑。数学上可以，但通常会过度平滑，因为 `q_j` 本身已经在表达市场分布；再给每个 anchor 加完整市场方差，可能会牺牲 bid/ask fit，或者让局部 skew/smile 被抹得太宽。

所以 `eta` 是一个 fit-versus-smoothness 参数：

- `eta = 0`：最接近线性插值，贴合能力强，但不光滑；
- 小 `eta`：轻微平滑；
- 中等 `eta`：平滑和贴合之间折中；
- 大 `eta`：更光滑，但可能过度平滑。

论文示例里常用 `eta = 0.25` 作为默认折中。我们先沿用这个值，后面可以通过比较 fit error、density smoothness 和 bid/ask 覆盖率来调参。

### 矩阵化

对固定 expiry `T_j`，可以预先构造 kernel matrix：

`G_j[l, i] = Call(k_i, k_l, eta * V_j)`

其中：

- `k_l` 是市场报价所在的 normalized strike；
- `k_i` 是模型 anchor grid；
- `q_j^i` 是后续线性规划要解的离散密度权重。

那么模型价格向量就是：

`c_hat_j = G_j @ q_j`

一旦 `G_j` 固定，后面对 `q_j` 的拟合就是线性的。这就是第四部分要完成的事情：只构造 kernel 输入，不求解 `q_j`。

In [ ]:
ETA = 0.25
MIN_QUOTES_PER_EXPIRY = 8

ETA

### Black-Scholes kernel 函数

这里使用 normalized 坐标下的 Black-Scholes call：

`Call(s, k, v)`

其中 `s` 是 anchor strike `k_i`，`k` 是市场 strike `k_l`，`v = eta * V_j` 是 total variance。注意这里的 `v` 已经是 total variance，不是 annualized volatility。

In [ ]:
def bs_call_kernel(spot: np.ndarray, strike: np.ndarray, total_variance: float) -> np.ndarray:
    spot = np.asarray(spot, dtype=float)
    strike = np.asarray(strike, dtype=float)
    if total_variance <= 0:
        return np.maximum(spot - strike, 0.0)

    sqrt_v = np.sqrt(total_variance)
    d1 = (np.log(spot / strike) + 0.5 * total_variance) / sqrt_v
    d2 = d1 - sqrt_v
    return spot * norm.cdf(d1) - strike * norm.cdf(d2)


def build_kernel_matrix(anchor_k: np.ndarray, market_k: np.ndarray, total_variance: float) -> np.ndarray:
    return bs_call_kernel(
        spot=anchor_k[None, :],
        strike=market_k[:, None],
        total_variance=total_variance,
    )

### 为每个 expiry 构造 market grid、anchor grid 和 kernel

第一版先采用最直接的 anchor grid：每个 expiry 的 model anchors 就取该 expiry 的市场 normalized strikes。

这不是最终生产版的唯一选择。后面可以加入：

- 统一的 global anchor grid；
- 更密的插值 grid；
- 左右 wing 的额外 boundary anchors；
- 不同 expiry 使用不同 model grid 的 production 实现。

但作为第一步，`anchor_k = market_k` 最透明，便于验证公式和矩阵维度。

In [ ]:
sanos_expiry_inputs = {}

for exdate, one in sanos_quotes.groupby("exdate", sort=True):
    one = one.sort_values("k", kind="mergesort").reset_index(drop=True)
    one = one.dropna(subset=["k", "pure_bid", "pure_mid", "pure_ask", "atm_total_variance"])
    if len(one) < MIN_QUOTES_PER_EXPIRY:
        continue

    market_k = one["k"].to_numpy(dtype=float)
    anchor_k = np.unique(market_k)
    V_j = float(one["atm_total_variance"].iloc[0])
    smooth_variance = ETA * V_j
    G_j = build_kernel_matrix(anchor_k=anchor_k, market_k=market_k, total_variance=smooth_variance)

    sanos_expiry_inputs[pd.Timestamp(exdate)] = {
        "dte": int(one["dte"].iloc[0]),
        "T": float(one["T"].iloc[0]),
        "V_j": V_j,
        "smooth_variance": smooth_variance,
        "market_k": market_k,
        "anchor_k": anchor_k,
        "pure_bid": one["pure_bid"].to_numpy(dtype=float),
        "pure_mid": one["pure_mid"].to_numpy(dtype=float),
        "pure_ask": one["pure_ask"].to_numpy(dtype=float),
        "G": G_j,
        "quotes": one,
    }

kernel_summary = pd.DataFrame(
    [
        {
            "exdate": exdate,
            "dte": item["dte"],
            "T": item["T"],
            "n_market": len(item["market_k"]),
            "n_anchor": len(item["anchor_k"]),
            "min_k": item["market_k"].min(),
            "max_k": item["market_k"].max(),
            "V_j": item["V_j"],
            "smooth_variance": item["smooth_variance"],
            "kernel_shape": item["G"].shape,
        }
        for exdate, item in sanos_expiry_inputs.items()
    ]
)

kernel_summary.head(15)

### Kernel sanity check

下面选一个中等期限的 expiry，检查 kernel matrix 的维度，并画出几个 anchor 生成的 Black-Scholes basis call。每一列都是一个 anchor `k_i` 对所有 market strikes `k_l` 的贡献：

`G_j[:, i] = Call(k_i, k_l, eta * V_j)`

后续 LP 要做的事情就是找到非负权重 `q_j`，让这些 basis call 的加权和落在市场 bid/ask 附近。

In [ ]:
example_exdate = kernel_summary.sort_values("dte").iloc[min(10, len(kernel_summary) - 1)]["exdate"]
example = sanos_expiry_inputs[pd.Timestamp(example_exdate)]

example_info = {
    "exdate": pd.Timestamp(example_exdate),
    "dte": example["dte"],
    "G_shape": example["G"].shape,
    "V_j": example["V_j"],
    "smooth_variance": example["smooth_variance"],
    "min_market_k": example["market_k"].min(),
    "max_market_k": example["market_k"].max(),
}

example_info

In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=example["market_k"],
        y=example["pure_mid"],
        mode="markers",
        name="market pure mid",
        marker=dict(size=5),
    )
)

anchor_indices = np.linspace(0, len(example["anchor_k"]) - 1, 7, dtype=int)
for idx in anchor_indices:
    fig.add_trace(
        go.Scatter(
            x=example["market_k"],
            y=example["G"][:, idx],
            mode="lines",
            name=f"basis anchor k={example['anchor_k'][idx]:.3f}",
        )
    )

fig.update_layout(
    height=650,
    title=f"Kernel basis sanity check: {pd.Timestamp(example_exdate).date()} ({example['dte']} DTE), eta={ETA}",
    xaxis_title="market normalized strike k",
    yaxis_title="pure call price / basis value",
)
fig.show()

### 到这里为止得到什么

现在每个 expiry 都有了：

- `market_k`
- `anchor_k`
- `pure_bid / pure_mid / pure_ask`
- `V_j`
- `G_j = Call(anchor_k, market_k, eta * V_j)`

下一步才是线性规划：

`G_j @ q_j ≈ pure call prices`

并给 `q_j` 加上：

`q_j >= 0`

`sum(q_j) = 1`

`sum(q_j * anchor_k) = 1`

以及跨 expiry 的 calendar no-arbitrage 约束。

## 第五部分：单 expiry SANOS LP demo

现在我们已经有了每个 expiry 的 kernel matrix：

`G @ q`

实际下一步就是求权重 `q`。先从单个 expiry 做 demo，不急着加入跨 expiry 的 calendar no-arbitrage。

固定一个 expiry 后，未知量是：

`q_i`

模型价格是：

`fit = G @ q`

约束是：

`q_i >= 0`

`sum_i q_i = 1`

`sum_i q_i * anchor_k_i = 1`

前两个让 `q` 是概率分布，第三个让 normalized underlying 的均值等于 1。

市场报价有 bid/ask spread，所以这里先用 soft bid/ask 约束：允许模型价格少量低于 bid 或高于 ask，但为这些超出部分付出惩罚。

In [ ]:
from scipy.optimize import linprog
from scipy import sparse

def normalized_spread_weights(bid: np.ndarray, ask: np.ndarray, floor_quantile: float = 0.10, cap: float = 100.0) -> np.ndarray:
    """Inverse-spread weights normalized within one expiry.

    The paper suggests inverse bid/ask spread as a natural weight. In raw form,
    however, very tight quotes can make the LP objective poorly scaled. We keep
    the same ordering idea but normalize the weights so the median active weight
    is around one, then cap extreme points.
    """
    bid = np.asarray(bid, dtype=float)
    ask = np.asarray(ask, dtype=float)
    spread = ask - bid
    finite_spread = spread[np.isfinite(spread) & (spread > 0)]
    if len(finite_spread) == 0:
        return np.ones_like(spread, dtype=float)

    spread_floor = np.quantile(finite_spread, floor_quantile)
    effective_spread = np.maximum(spread, spread_floor)
    raw_weights = 1.0 / effective_spread

    finite_weights = raw_weights[np.isfinite(raw_weights) & (raw_weights > 0)]
    scale = np.median(finite_weights) if len(finite_weights) else 1.0
    if not np.isfinite(scale) or scale <= 0:
        scale = 1.0

    weights = raw_weights / scale
    weights = np.clip(weights, 1.0 / cap, cap)
    return weights


### LP 形式

引入三个非负 slack：

`u_l >= 0`：模型价格高于 ask 的超出量；

`v_l >= 0`：模型价格低于 bid 的超出量；

`a_l >= 0`：模型价格偏离 mid 的绝对误差。

对每个市场报价点 `l`：

`Gq <= ask + u`

`Gq >= bid - v`

`a_l >= |Gq - mid|`

目标函数：

`min sum_l w_l * (u_l + v_l + epsilon * a_l)`

其中：

`w_l = 1 / (pure_ask_l - pure_bid_l)`

`epsilon = 1e-8`

bid/ask 越窄，权重越高。这个 LP 如果能找到 `u=v=0`，说明该 expiry 能完全落在 bid/ask 内；`epsilon * a_l` 是论文推荐的极弱 mid tie-breaker，只在多个 bid/ask 内解同样好时帮 solver 选择更靠近 mid 的解。

In [ ]:
MID_TIE_BREAK_EPS = 1e-8


def solve_single_expiry_lp(expiry_input: dict) -> dict:
    G = expiry_input["G"]
    anchor_k = expiry_input["anchor_k"]
    bid = expiry_input["pure_bid"]
    mid = expiry_input["pure_mid"]
    ask = expiry_input["pure_ask"]

    n_quotes, n_anchor = G.shape
    spread = np.maximum(ask - bid, 1e-10)
    weights = 1.0 / spread

    # Variables are [q_0..q_{n-1}, u_0..u_{m-1}, v_0..v_{m-1}, a_0..a_{m-1}].
    n_vars = n_anchor + 3 * n_quotes
    q_slice = slice(0, n_anchor)
    u_slice = slice(n_anchor, n_anchor + n_quotes)
    v_slice = slice(n_anchor + n_quotes, n_anchor + 2 * n_quotes)
    a_slice = slice(n_anchor + 2 * n_quotes, n_vars)

    c = np.zeros(n_vars)
    c[u_slice] = weights
    c[v_slice] = weights
    c[a_slice] = MID_TIE_BREAK_EPS * weights

    A_eq = np.zeros((2, n_vars))
    A_eq[0, q_slice] = 1.0
    A_eq[1, q_slice] = anchor_k
    b_eq = np.array([1.0, 1.0])

    A_ub = []
    b_ub = []

    # Gq - u <= ask
    upper = np.zeros((n_quotes, n_vars))
    upper[:, q_slice] = G
    upper[:, u_slice] = -np.eye(n_quotes)
    A_ub.append(upper)
    b_ub.append(ask)

    # -Gq - v <= -bid
    lower = np.zeros((n_quotes, n_vars))
    lower[:, q_slice] = -G
    lower[:, v_slice] = -np.eye(n_quotes)
    A_ub.append(lower)
    b_ub.append(-bid)

    # Gq - a <= mid and -Gq - a <= -mid. This is the paper's tiny
    # epsilon * |fit - mid| tie-breaker inside the bid/ask band.
    mid_upper = np.zeros((n_quotes, n_vars))
    mid_upper[:, q_slice] = G
    mid_upper[:, a_slice] = -np.eye(n_quotes)
    A_ub.append(mid_upper)
    b_ub.append(mid)

    mid_lower = np.zeros((n_quotes, n_vars))
    mid_lower[:, q_slice] = -G
    mid_lower[:, a_slice] = -np.eye(n_quotes)
    A_ub.append(mid_lower)
    b_ub.append(-mid)

    bounds = [(0.0, None)] * n_vars

    result = linprog(
        c=c,
        A_ub=np.vstack(A_ub),
        b_ub=np.concatenate(b_ub),
        A_eq=A_eq,
        b_eq=b_eq,
        bounds=bounds,
        method="highs",
    )

    if not result.success:
        return {"success": False, "message": result.message, "result": result}

    x = result.x
    q = x[q_slice]
    u = x[u_slice]
    v = x[v_slice]
    a = x[a_slice]
    fit = G @ q

    return {
        "success": True,
        "message": result.message,
        "result": result,
        "q": q,
        "u": u,
        "v": v,
        "a": a,
        "fit": fit,
        "objective": result.fun,
        "sum_q": q.sum(),
        "mean_q": np.dot(q, anchor_k),
        "max_outside_bid_ask": max(u.max(initial=0.0), v.max(initial=0.0)),
        "weighted_outside_error": np.sum(weights * (u + v)),
        "weighted_mid_tiebreak_error": MID_TIE_BREAK_EPS * np.sum(weights * a),
        "mid_rmse": np.sqrt(np.mean((fit - mid) ** 2)),
        "bid_ask_coverage": np.mean((fit >= bid - 1e-12) & (fit <= ask + 1e-12)),
    }

### 选择一个 30D 左右 expiry 做 demo

先选择 DTE 最接近 30 的 expiry。这个期限通常报价点较多，也不会像 0DTE 那样受极端微结构噪音影响。

In [ ]:
target_dte = 30
demo_exdate = kernel_summary.iloc[(kernel_summary["dte"] - target_dte).abs().argmin()]["exdate"]
demo_input = sanos_expiry_inputs[pd.Timestamp(demo_exdate)]
demo_solution = solve_single_expiry_lp(demo_input)

demo_summary = {
    "exdate": pd.Timestamp(demo_exdate),
    "dte": demo_input["dte"],
    "success": demo_solution["success"],
    "message": demo_solution["message"],
    "n_quotes": len(demo_input["market_k"]),
    "n_anchor": len(demo_input["anchor_k"]),
    "objective": demo_solution.get("objective"),
    "sum_q": demo_solution.get("sum_q"),
    "mean_q": demo_solution.get("mean_q"),
    "max_outside_bid_ask": demo_solution.get("max_outside_bid_ask"),
    "weighted_outside_error": demo_solution.get("weighted_outside_error"),
    "mid_rmse": demo_solution.get("mid_rmse"),
    "bid_ask_coverage": demo_solution.get("bid_ask_coverage"),
}

demo_summary

### 拟合结果图

下面比较：

- market pure bid/ask；
- market pure mid；
- LP 拟合曲线 `G @ q`。

如果 fitted curve 大部分落在 bid/ask 内，说明单 expiry 的 SANOS 表达已经工作正常。

In [ ]:
demo_quotes = demo_input["quotes"].copy()
demo_quotes["fit"] = demo_solution["fit"]
demo_quotes["outside_high"] = np.maximum(demo_quotes["fit"] - demo_quotes["pure_ask"], 0.0)
demo_quotes["outside_low"] = np.maximum(demo_quotes["pure_bid"] - demo_quotes["fit"], 0.0)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=demo_quotes["k"],
        y=demo_quotes["pure_bid"],
        mode="lines",
        name="pure bid",
        line=dict(width=1),
    )
)
fig.add_trace(
    go.Scatter(
        x=demo_quotes["k"],
        y=demo_quotes["pure_ask"],
        mode="lines",
        name="pure ask",
        fill="tonexty",
        line=dict(width=1),
    )
)
fig.add_trace(
    go.Scatter(
        x=demo_quotes["k"],
        y=demo_quotes["pure_mid"],
        mode="markers",
        name="pure mid",
        marker=dict(size=4),
    )
)
fig.add_trace(
    go.Scatter(
        x=demo_quotes["k"],
        y=demo_quotes["fit"],
        mode="lines",
        name="SANOS single-expiry fit",
        line=dict(width=3),
    )
)

fig.update_layout(
    height=650,
    title=f"Single-expiry SANOS LP fit: {pd.Timestamp(demo_exdate).date()} ({demo_input['dte']} DTE), eta={ETA}",
    xaxis_title="normalized strike k",
    yaxis_title="pure call price",
)
fig.show()

### q 分布诊断

`q` 是 LP 求出来的离散 anchor 权重。它不是直接二阶导得到的唯一密度，而是在当前 kernel、bid/ask、约束下选出来的一组可行权重。

这里检查它是否非负、是否主要集中在合理的 moneyness 区域，以及是否满足 `sum(q)=1` 和 `sum(q*k_i)=1`。

In [ ]:
q_diagnostics = pd.DataFrame({
    "anchor_k": demo_input["anchor_k"],
    "q": demo_solution["q"],
})
q_diagnostics["cumulative_q"] = q_diagnostics["q"].cumsum()

q_diagnostics_summary = {
    "min_q": q_diagnostics["q"].min(),
    "max_q": q_diagnostics["q"].max(),
    "nonzero_q_count": int((q_diagnostics["q"] > 1e-10).sum()),
    "sum_q": q_diagnostics["q"].sum(),
    "mean_q": np.dot(q_diagnostics["q"], q_diagnostics["anchor_k"]),
}

q_diagnostics_summary

In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=q_diagnostics["anchor_k"],
        y=q_diagnostics["q"],
        name="q weight",
    )
)
fig.update_layout(
    height=550,
    title=f"Single-expiry q distribution: {pd.Timestamp(demo_exdate).date()} ({demo_input['dte']} DTE)",
    xaxis_title="anchor normalized strike k_i",
    yaxis_title="q_i",
)
fig.show()

### 下一步

如果单 expiry LP 的 fit 合理，下一步有两条路：

1. 对每个 expiry 独立求 `q_j`，检查每条曲线的 fit 和 density；
2. 构造一个多 expiry 联合 LP，加入 calendar no-arbitrage：

`C_hat(T_{j+1}, k) >= C_hat(T_j, k)`

论文 production model 的完整价值主要在第二步，也就是全局联合拟合并严格控制 time/strike 方向无套利。

## 第六部分：多 expiry 联合 LP

单 expiry LP 只保证每条 call curve 自己在 strike 方向上是无套利的。论文 production model 的关键是把多个 expiry 一起求解，并加入 calendar no-arbitrage：

`C_hat(T_{j+1}, k) >= C_hat(T_j, k)`

这一步把多个 expiry 的 `q_j` 同时作为 LP 变量：

`q_1, q_2, ..., q_M`

并对每个 expiry 保留：

`q_j >= 0`

`sum_i q_j^i = 1`

`sum_i q_j^i * k_i = 1`

同时对每个 expiry 的市场报价加入 soft bid/ask slack。

### Calendar 约束怎么做

不同 expiry 的 market strikes 和 anchor grid 可能不同，所以不能简单逐点相减。这里在每一对相邻 expiry 上构造一个 common calendar check grid：

`calendar_k = union(market_k_j, market_k_{j+1}, dense_linspace)`

全量 expiry 联合求解时，只在 market union grid 上约束 calendar 仍可能在相邻 strike 之间出现小违例。当前 notebook 对每对相邻 expiry 使用 market union 加一条 `CALENDAR_DENSE_GRID_POINTS = 801` 的线性密网格，并把每一对约束都扩展到全局 observed fitting domain，用于更接近 dense-grid 的 hard calendar 检查。后续生产版可以继续提高这个数，或者使用更结构化的 model grid。

然后分别在这个 common grid 上重新计算 evaluation kernel：

`G_eval_j = Call(anchor_k_j, calendar_k, eta * V_j)`

`G_eval_{j+1} = Call(anchor_k_{j+1}, calendar_k, eta * V_{j+1})`

calendar no-arbitrage 写成线性约束：

`G_eval_j @ q_j - G_eval_{j+1} @ q_{j+1} <= 0`

也就是长期限价格不能低于短期限价格。

全量求解时，市场报价点仍然全部保留；但 model anchor grid 可以比 market grid 更稀疏。当前 notebook 使用 smarter model grid：ATM 附近更密，wings 用分位数抽样，长端允许更多 anchors，并在两端加入轻微 boundary anchors。这和论文 implementation section 里“market strikes 与 model strikes 可以不同”的思路一致。

In [ ]:
JOINT_MAX_EXPIRIES = None
JOINT_MAX_ANCHORS_PER_EXPIRY = 80
CALENDAR_DENSE_GRID_POINTS = 801
CALENDAR_TOL = 1e-7

ordered_exdates = kernel_summary.sort_values("dte")["exdate"]
joint_exdates = ordered_exdates.tolist() if JOINT_MAX_EXPIRIES is None else ordered_exdates.head(JOINT_MAX_EXPIRIES).tolist()
joint_inputs = {pd.Timestamp(exdate): sanos_expiry_inputs[pd.Timestamp(exdate)] for exdate in joint_exdates}

def max_anchors_for_dte(dte: int) -> int:
    if dte <= 30:
        return 90
    if dte <= 90:
        return 110
    return 150


def make_model_anchor_grid(market_k: np.ndarray, dte: int, atm_width: float = 0.15) -> np.ndarray:
    market_k = np.unique(np.asarray(market_k, dtype=float))
    market_k = market_k[np.isfinite(market_k)]
    if len(market_k) == 0:
        return market_k

    max_anchors = max_anchors_for_dte(int(dte))
    if len(market_k) <= max_anchors:
        base = market_k
    else:
        atm_mask = np.abs(market_k - 1.0) <= atm_width
        atm = market_k[atm_mask]
        left = market_k[market_k < 1.0 - atm_width]
        right = market_k[market_k > 1.0 + atm_width]

        # Keep ATM relatively dense because it drives fit and variance diagnostics.
        atm_budget = min(len(atm), max(20, int(max_anchors * 0.55)))
        wing_budget = max_anchors - atm_budget
        left_budget = min(len(left), max(2, wing_budget // 2))
        right_budget = min(len(right), max(2, wing_budget - left_budget))

        def take_quantile_grid(values: np.ndarray, budget: int) -> np.ndarray:
            if len(values) == 0 or budget <= 0:
                return np.array([], dtype=float)
            if len(values) <= budget:
                return values
            idx = np.linspace(0, len(values) - 1, budget).round().astype(int)
            return values[np.unique(idx)]

        atm_grid = take_quantile_grid(atm, atm_budget)
        left_grid = take_quantile_grid(left, left_budget)
        right_grid = take_quantile_grid(right, right_budget)
        base = np.unique(np.r_[left_grid, atm_grid, right_grid, market_k[0], market_k[-1]])

    # Add mild boundary anchors outside observed market range to give tails room.
    left_boundary = max(base[0] * 0.95, 1e-6)
    right_boundary = base[-1] * 1.05
    return np.unique(np.r_[left_boundary, base, right_boundary])


def thin_anchor_grid(market_k: np.ndarray, max_anchors=None) -> np.ndarray:
    market_k = np.unique(np.asarray(market_k, dtype=float))
    if max_anchors is None or len(market_k) <= max_anchors:
        return market_k
    positions = np.linspace(0, len(market_k) - 1, max_anchors).round().astype(int)
    atm_position = np.argmin(np.abs(market_k - 1.0))
    positions = np.unique(np.r_[positions, atm_position, 0, len(market_k) - 1])
    return market_k[positions]

reduced_joint_inputs = {}
for exdate, item in joint_inputs.items():
    reduced_item = item.copy()
    if JOINT_MAX_ANCHORS_PER_EXPIRY is None:
        reduced_anchor_k = make_model_anchor_grid(item["market_k"], item["dte"])
    else:
        reduced_anchor_k = thin_anchor_grid(item["market_k"], JOINT_MAX_ANCHORS_PER_EXPIRY)
    reduced_item["anchor_k"] = reduced_anchor_k
    reduced_item["G"] = build_kernel_matrix(reduced_anchor_k, item["market_k"], item["smooth_variance"])
    reduced_joint_inputs[exdate] = reduced_item
joint_inputs = reduced_joint_inputs
GLOBAL_CALENDAR_K_MIN = min(float(np.min(item["market_k"])) for item in joint_inputs.values())
GLOBAL_CALENDAR_K_MAX = max(float(np.max(item["market_k"])) for item in joint_inputs.values())

[(pd.Timestamp(exdate).date(), joint_inputs[pd.Timestamp(exdate)]["dte"], len(joint_inputs[pd.Timestamp(exdate)]["market_k"])) for exdate in joint_exdates]

In [ ]:
def thin_calendar_grid(calendar_k: np.ndarray, dense_points=None, k_min=None, k_max=None) -> np.ndarray:
    calendar_k = np.asarray(calendar_k, dtype=float)
    calendar_k = np.unique(calendar_k[np.isfinite(calendar_k)])
    if dense_points is None or len(calendar_k) == 0:
        return calendar_k
    left = calendar_k[0] if k_min is None else float(k_min)
    right = calendar_k[-1] if k_max is None else float(k_max)
    dense = np.linspace(left, right, int(dense_points))
    return np.unique(np.r_[calendar_k, dense])


def solve_joint_expiry_lp(joint_inputs: dict) -> dict:
    exdates = list(joint_inputs.keys())
    q_offsets = {}
    u_offsets = {}
    v_offsets = {}
    a_offsets = {}
    cursor = 0

    for exdate in exdates:
        item = joint_inputs[exdate]
        n_anchor = len(item["anchor_k"])
        n_quotes = len(item["market_k"])
        q_offsets[exdate] = (cursor, cursor + n_anchor)
        cursor += n_anchor
        u_offsets[exdate] = (cursor, cursor + n_quotes)
        cursor += n_quotes
        v_offsets[exdate] = (cursor, cursor + n_quotes)
        cursor += n_quotes
        a_offsets[exdate] = (cursor, cursor + n_quotes)
        cursor += n_quotes

    n_vars = cursor
    c = np.zeros(n_vars)
    bounds = [(0.0, None)] * n_vars

    eq_rows, eq_cols, eq_data, b_eq = [], [], [], []
    ub_rows, ub_cols, ub_data, b_ub = [], [], [], []
    eq_r = 0
    ub_r = 0

    def add_eq(indices, values, rhs):
        nonlocal eq_r
        eq_rows.extend([eq_r] * len(indices))
        eq_cols.extend(indices)
        eq_data.extend(values)
        b_eq.append(rhs)
        eq_r += 1

    def add_ub(indices, values, rhs):
        nonlocal ub_r
        ub_rows.extend([ub_r] * len(indices))
        ub_cols.extend(indices)
        ub_data.extend(values)
        b_ub.append(rhs)
        ub_r += 1

    for exdate in exdates:
        item = joint_inputs[exdate]
        G = item["G"]
        anchor_k = item["anchor_k"]
        bid = item["pure_bid"]
        ask = item["pure_ask"]
        weights = normalized_spread_weights(bid, ask)
        n_quotes, n_anchor = G.shape
        q0, q1 = q_offsets[exdate]
        u0, u1 = u_offsets[exdate]
        v0, v1 = v_offsets[exdate]
        a0, a1 = a_offsets[exdate]
        q_idx = np.arange(q0, q1)

        c[u0:u1] = weights
        c[v0:v1] = weights
        c[a0:a1] = MID_TIE_BREAK_EPS * weights

        add_eq(q_idx.tolist(), np.ones(n_anchor).tolist(), 1.0)
        add_eq(q_idx.tolist(), anchor_k.tolist(), 1.0)

        for l in range(n_quotes):
            # Gq - u <= ask
            add_ub(
                np.r_[q_idx, u0 + l].tolist(),
                np.r_[G[l, :], -1.0].tolist(),
                ask[l],
            )
            # -Gq - v <= -bid
            add_ub(
                np.r_[q_idx, v0 + l].tolist(),
                np.r_[-G[l, :], -1.0].tolist(),
                -bid[l],
            )
            # Gq - a <= mid and -Gq - a <= -mid: tiny mid tie-breaker.
            add_ub(
                np.r_[q_idx, a0 + l].tolist(),
                np.r_[G[l, :], -1.0].tolist(),
                item["pure_mid"][l],
            )
            add_ub(
                np.r_[q_idx, a0 + l].tolist(),
                np.r_[-G[l, :], -1.0].tolist(),
                -item["pure_mid"][l],
            )

    calendar_constraint_count = 0
    for prev_exdate, next_exdate in zip(exdates[:-1], exdates[1:]):
        prev_item = joint_inputs[prev_exdate]
        next_item = joint_inputs[next_exdate]
        calendar_k = thin_calendar_grid(
            np.union1d(prev_item["market_k"], next_item["market_k"]),
            CALENDAR_DENSE_GRID_POINTS,
            GLOBAL_CALENDAR_K_MIN,
            GLOBAL_CALENDAR_K_MAX,
        )
        prev_eval = build_kernel_matrix(
            anchor_k=prev_item["anchor_k"],
            market_k=calendar_k,
            total_variance=prev_item["smooth_variance"],
        )
        next_eval = build_kernel_matrix(
            anchor_k=next_item["anchor_k"],
            market_k=calendar_k,
            total_variance=next_item["smooth_variance"],
        )
        p0, p1 = q_offsets[prev_exdate]
        n0, n1 = q_offsets[next_exdate]
        p_idx = np.arange(p0, p1)
        n_idx = np.arange(n0, n1)

        for l in range(len(calendar_k)):
            add_ub(
                np.r_[p_idx, n_idx].tolist(),
                np.r_[prev_eval[l, :], -next_eval[l, :]].tolist(),
                0.0,
            )
            calendar_constraint_count += 1

    A_eq = sparse.coo_matrix((eq_data, (eq_rows, eq_cols)), shape=(eq_r, n_vars)).tocsr()
    A_ub = sparse.coo_matrix((ub_data, (ub_rows, ub_cols)), shape=(ub_r, n_vars)).tocsr()

    result = linprog(
        c=c,
        A_ub=A_ub,
        b_ub=np.asarray(b_ub),
        A_eq=A_eq,
        b_eq=np.asarray(b_eq),
        bounds=bounds,
        method="highs",
        options={"presolve": True},
    )

    solution = {
        "success": result.success,
        "message": result.message,
        "result": result,
        "exdates": exdates,
        "q_offsets": q_offsets,
        "u_offsets": u_offsets,
        "v_offsets": v_offsets,
        "a_offsets": a_offsets,
        "calendar_constraint_count": calendar_constraint_count,
        "n_vars": n_vars,
        "n_eq": A_eq.shape[0],
        "n_ub": A_ub.shape[0],
        "A_eq_nnz": A_eq.nnz,
        "A_ub_nnz": A_ub.nnz,
    }

    if not result.success:
        return solution

    x = result.x
    per_expiry = []
    q_by_expiry = {}
    fit_by_expiry = {}
    for exdate in exdates:
        item = joint_inputs[exdate]
        q0, q1 = q_offsets[exdate]
        u0, u1 = u_offsets[exdate]
        v0, v1 = v_offsets[exdate]
        a0, a1 = a_offsets[exdate]
        q = x[q0:q1]
        u = x[u0:u1]
        v = x[v0:v1]
        a = x[a0:a1]
        fit = item["G"] @ q
        bid = item["pure_bid"]
        ask = item["pure_ask"]
        mid = item["pure_mid"]
        weights = normalized_spread_weights(bid, ask)
        q_by_expiry[exdate] = q
        fit_by_expiry[exdate] = fit
        per_expiry.append(
            {
                "exdate": exdate,
                "dte": item["dte"],
                "n_quotes": len(item["market_k"]),
                "n_anchor": len(item["anchor_k"]),
                "sum_q": q.sum(),
                "mean_q": np.dot(q, item["anchor_k"]),
                "nonzero_q_count": int((q > 1e-10).sum()),
                "max_outside_bid_ask": max(u.max(initial=0.0), v.max(initial=0.0)),
                "weighted_outside_error": np.sum(weights * (u + v)),
                "weighted_mid_tiebreak_error": MID_TIE_BREAK_EPS * np.sum(weights * a),
                "mid_rmse": np.sqrt(np.mean((fit - mid) ** 2)),
                "bid_ask_coverage": np.mean((fit >= bid - 1e-9) & (fit <= ask + 1e-9)),
            }
        )

    calendar_checks = []
    for prev_exdate, next_exdate in zip(exdates[:-1], exdates[1:]):
        prev_item = joint_inputs[prev_exdate]
        next_item = joint_inputs[next_exdate]
        calendar_k = thin_calendar_grid(
            np.union1d(prev_item["market_k"], next_item["market_k"]),
            CALENDAR_DENSE_GRID_POINTS,
            GLOBAL_CALENDAR_K_MIN,
            GLOBAL_CALENDAR_K_MAX,
        )
        prev_eval = build_kernel_matrix(prev_item["anchor_k"], calendar_k, prev_item["smooth_variance"]) @ q_by_expiry[prev_exdate]
        next_eval = build_kernel_matrix(next_item["anchor_k"], calendar_k, next_item["smooth_variance"]) @ q_by_expiry[next_exdate]
        diff = next_eval - prev_eval
        calendar_checks.append(
            {
                "prev_exdate": prev_exdate,
                "next_exdate": next_exdate,
                "prev_dte": prev_item["dte"],
                "next_dte": next_item["dte"],
                "n_check_k": len(calendar_k),
                "min_calendar_diff": diff.min(),
                "violation_count": int((diff < -CALENDAR_TOL).sum()),
            }
        )

    solution.update(
        {
            "x": x,
            "q_by_expiry": q_by_expiry,
            "fit_by_expiry": fit_by_expiry,
            "per_expiry": pd.DataFrame(per_expiry),
            "calendar_checks": pd.DataFrame(calendar_checks),
            "objective": result.fun,
        }
    )
    return solution

In [ ]:
joint_solution = solve_joint_expiry_lp(joint_inputs)

joint_summary = {
    "success": joint_solution["success"],
    "message": joint_solution["message"],
    "objective": joint_solution.get("objective"),
    "n_expiries": len(joint_solution["exdates"]),
    "n_vars": joint_solution["n_vars"],
    "n_eq": joint_solution["n_eq"],
    "n_ub": joint_solution["n_ub"],
    "calendar_constraint_count": joint_solution["calendar_constraint_count"],
    "calendar_dense_grid_points": CALENDAR_DENSE_GRID_POINTS,
    "global_calendar_k_min": GLOBAL_CALENDAR_K_MIN,
    "global_calendar_k_max": GLOBAL_CALENDAR_K_MAX,
    "fit_min_volume": FIT_MIN_VOLUME,
    "fit_min_normalized_k": FIT_MIN_NORMALIZED_K,
}

joint_summary

### 联合 LP 拟合诊断

这里看每个 expiry 的拟合质量和约束满足情况。重点关注：

- `sum_q` 是否等于 1；
- `mean_q` 是否等于 1；
- `bid_ask_coverage` 是否接近 1；
- `weighted_outside_error` 是否很小；
- `mid_rmse` 是否合理。

In [ ]:
if joint_solution["success"]:
    joint_per_expiry = joint_solution["per_expiry"]
else:
    joint_per_expiry = pd.DataFrame()

joint_per_expiry

### Calendar no-arbitrage 诊断

这里检查相邻 expiry 在 common calendar grid 上的差：

`C_hat(T_{j+1}, k) - C_hat(T_j, k)`

理论上应当非负。`min_calendar_diff` 如果接近 0 或大于 0，说明 calendar 约束正常。

In [ ]:
if joint_solution["success"]:
    joint_calendar_checks = joint_solution["calendar_checks"]
else:
    joint_calendar_checks = pd.DataFrame()

joint_calendar_checks

In [ ]:
if joint_solution["success"]:
    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.08,
        subplot_titles=("Joint LP fit diagnostics", "Calendar constraint diagnostics"),
    )

    fig.add_trace(
        go.Scatter(
            x=joint_per_expiry["dte"],
            y=joint_per_expiry["mid_rmse"],
            mode="markers+lines",
            name="mid RMSE",
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=joint_per_expiry["dte"],
            y=joint_per_expiry["weighted_outside_error"],
            mode="markers+lines",
            name="weighted outside error",
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=joint_calendar_checks["next_dte"],
            y=joint_calendar_checks["min_calendar_diff"],
            mode="markers+lines",
            name="min calendar diff",
        ),
        row=2,
        col=1,
    )

    joint_label = "all expiries" if JOINT_MAX_EXPIRIES is None else f"first {JOINT_MAX_EXPIRIES} expiries"
    fig.update_xaxes(title_text="DTE", row=2, col=1)
    fig.update_yaxes(title_text="fit diagnostic", row=1, col=1)
    fig.update_yaxes(title_text="min C(next)-C(prev)", row=2, col=1)
    fig.update_layout(height=750, title=f"Joint SANOS LP diagnostics, {joint_label}")
    fig.show()
else:
    print(joint_solution["message"])

### 联合 LP 拟合曲线示例

下面画几个 expiry 的联合 LP 拟合结果，检查在加入 calendar 约束后，每条 smile 是否仍然贴合 bid/ask。

In [ ]:
if joint_solution["success"]:
    fig = go.Figure()
    pick_positions = [0, min(3, len(joint_per_expiry) - 1), min(7, len(joint_per_expiry) - 1), len(joint_per_expiry) - 1]
    plot_exdates = joint_per_expiry.sort_values("dte").iloc[pick_positions]["exdate"].drop_duplicates()

    for exdate in plot_exdates:
        item = joint_inputs[pd.Timestamp(exdate)]
        quotes = item["quotes"].copy()
        quotes["fit"] = joint_solution["fit_by_expiry"][pd.Timestamp(exdate)]
        label = f"{pd.Timestamp(exdate).date()} ({item['dte']}D)"
        fig.add_trace(
            go.Scatter(
                x=quotes["k"],
                y=quotes["pure_mid"],
                mode="markers",
                marker=dict(size=4),
                name=f"mid {label}",
            )
        )
        fig.add_trace(
            go.Scatter(
                x=quotes["k"],
                y=quotes["fit"],
                mode="lines",
                name=f"fit {label}",
            )
        )

    joint_label = "all expiries" if JOINT_MAX_EXPIRIES is None else f"first {JOINT_MAX_EXPIRIES} expiries"
    fig.update_layout(
        height=700,
        title=f"Joint LP fitted curves, {joint_label}",
        xaxis_title="normalized strike k",
        yaxis_title="pure call price",
    )
    fig.show()
else:
    print(joint_solution["message"])

## 第七部分：三维 fitted surface 可视化

联合 LP 求出所有 expiry 的 `q_j` 后，就可以在规则网格上评价 fitted surface。

SANOS 本身拟合的是 normalized call price：

`C_hat(T_j, k) = G_eval_j(k) @ q_j`

所以第一张 3D 图画的是：

`x = normalized strike k`

`y = DTE`

`z = fitted pure call price`

第二张图把 fitted pure call price 再反解成 Black-Scholes implied vol，用来观察传统意义上的 volatility surface。

In [ ]:
if joint_solution["success"] and "q_by_expiry" in joint_solution:
    all_market_k = np.concatenate([item["market_k"] for item in joint_inputs.values()])
    surface_k_grid = np.linspace(
        float(np.nanquantile(all_market_k, 0.01)),
        float(np.nanquantile(all_market_k, 0.99)),
        151,
    )
else:
    surface_k_grid = np.asarray([])

pure_call_surface = []
otm_iv_surface = []
surface_dte = []
surface_exdates = []

if joint_solution["success"] and "q_by_expiry" in joint_solution:
    for exdate in joint_solution["exdates"]:
        exdate = pd.Timestamp(exdate)
        item = joint_inputs[exdate]
        q = joint_solution["q_by_expiry"][exdate]

        G_grid = build_kernel_matrix(
            market_k=surface_k_grid,
            anchor_k=item["anchor_k"],
            total_variance=item["smooth_variance"],
        )
        pure_fit = G_grid @ q
        pure_call_surface.append(pure_fit)
        surface_dte.append(item["dte"])
        surface_exdates.append(exdate)

        iv_row = []
        for k, pure_call_price in zip(surface_k_grid, pure_fit):
            cp_flag = otm_cp_flag_from_k(k)
            otm_price = otm_price_from_pure_call(k, pure_call_price)
            iv_row.append(implied_vol_from_pure_option(k, otm_price, item["T"], cp_flag))
        otm_iv_surface.append(iv_row)
else:
    print(joint_solution["message"])

pure_call_surface = np.asarray(pure_call_surface)
otm_iv_surface = np.asarray(otm_iv_surface)
surface_dte = np.asarray(surface_dte)

pure_call_surface.shape, otm_iv_surface.shape

### Fitted normalized call price surface

这是模型真正拟合出来的对象。它应该随 strike 单调下降、凸，并且在 calendar 方向上不下降。

In [ ]:
if pure_call_surface.size:
    fig = go.Figure(
        data=[
            go.Surface(
                x=surface_k_grid,
                y=surface_dte,
                z=pure_call_surface,
                colorscale="Viridis",
                colorbar=dict(title="pure call"),
            )
        ]
    )
    fig.update_layout(
        height=760,
        title=f"Fitted SANOS pure call surface: {PRODUCT_PREFIX} {TRADE_DATE.date()}",
        scene=dict(
            xaxis_title="normalized strike k",
            yaxis_title="DTE",
            zaxis_title="pure call price",
        ),
    )
    fig.show()
else:
    print("Joint LP did not produce q weights, so the pure call surface is skipped.")

### Fitted implied volatility surface

这张图是把 fitted pure call price 反解成 Black-Scholes IV 后得到的传统波动率曲面。注意：SANOS 的主拟合对象不是 IV，而是 call price；IV surface 只是展示和诊断。

In [ ]:
if otm_iv_surface.size:
    otm_iv_surface_plot = np.where(np.isfinite(otm_iv_surface), otm_iv_surface, np.nan)

    fig_iv_surface = go.Figure(
        data=[
            go.Surface(
                x=surface_k_grid,
                y=surface_dte,
                z=otm_iv_surface_plot,
                colorscale="Viridis",
                colorbar=dict(title="OTM IV"),
            )
        ]
    )
    fig_iv_surface.update_layout(
        title=f"Fitted SANOS OTM implied volatility surface: {PRODUCT_PREFIX} {TRADE_DATE.date()}",
        scene=dict(
            xaxis_title="normalized strike k",
            yaxis_title="DTE",
            zaxis_title="OTM implied volatility",
        ),
        width=1100,
        height=760,
    )
    fig_iv_surface.show()
else:
    print("Joint LP did not produce q weights, so the OTM IV surface is skipped.")

### 7.3 价格拟合与 OTM IV 诊断

这一张诊断分成两层：

第一层看 **pure call price**。这是 SANOS/LP 真正拟合的对象，因此价格图是判断 LP 是否贴近 bid/ask 的主诊断。

第二层看 **OTM implied volatility**。IV 只是把价格重新映射到波动率坐标；为了避免短端深 ITM option 反解时 vega 极低、数值极不稳定，这里采用市场上更常用的 OTM 口径：

- 当 `k >= 1` 时，用 call price 反解 IV；
- 当 `k < 1` 时，用 put price 反解 IV；
- 对模型 fitted call，左翼 put price 由 normalized parity 得到：`P = C + k - 1`；
- 对市场点，左翼直接使用市场 put，右翼直接使用市场 call。

因此，这张 IV 图不是新的拟合目标，而是价格拟合在交易员更熟悉的 OTM IV 坐标下的可视化检查。若价格拟合正常但 IV 图在极短端或极低 vega 区域剧烈跳动，优先相信价格诊断，并把这些低 vega 点从 IV 展示中排除。

In [ ]:
MAX_DISPLAY_IV = 3.0
VEGA_TOL = 1e-4

fig_price_slices = go.Figure()
fig_otm_iv_slices = go.Figure()

use_joint_slice_fit = joint_solution["success"] and "q_by_expiry" in joint_solution
fit_label_prefix = "joint LP" if use_joint_slice_fit else "single-expiry LP fallback"

sorted_exdates = sorted(pd.Timestamp(x) for x in joint_inputs.keys())
pick_positions = [0, min(7, len(sorted_exdates) - 1), min(22, len(sorted_exdates) - 1), min(37, len(sorted_exdates) - 1), len(sorted_exdates) - 1]
selected_exdates = []
for pos in pick_positions:
    exdate = sorted_exdates[pos]
    if exdate not in selected_exdates:
        selected_exdates.append(exdate)

slice_fit_by_expiry = {}
slice_q_by_expiry = {}

for exdate in selected_exdates:
    item = joint_inputs[exdate]
    if use_joint_slice_fit:
        q = joint_solution["q_by_expiry"][exdate]
    else:
        single_solution = solve_single_expiry_lp(item)
        if not single_solution["success"]:
            print(f"Skip {exdate.date()}: {single_solution['message']}")
            continue
        q = single_solution["q"]

    slice_q_by_expiry[exdate] = q
    pure_fit = build_kernel_matrix(
        market_k=surface_k_grid,
        anchor_k=item["anchor_k"],
        total_variance=item["smooth_variance"],
    ) @ q
    slice_fit_by_expiry[exdate] = pure_fit

    quotes = item["quotes"].copy().sort_values("k")
    fig_price_slices.add_trace(
        go.Scatter(
            x=quotes["k"],
            y=quotes["pure_mid"],
            mode="markers",
            marker=dict(size=4),
            name=f"market call {exdate.date()} ({item['dte']}D)",
        )
    )
    fig_price_slices.add_trace(
        go.Scatter(
            x=surface_k_grid,
            y=pure_fit,
            mode="lines",
            name=f"{fit_label_prefix} call {exdate.date()} ({item['dte']}D)",
        )
    )

    market_otm = option_chain[
        (option_chain["exdate"] == exdate)
        & (
            ((option_chain["cp_flag"] == "C") & (option_chain["k"] >= 1.0))
            | ((option_chain["cp_flag"] == "P") & (option_chain["k"] < 1.0))
        )
    ].copy()
    market_otm["otm_iv"] = [
        implied_vol_from_pure_option(k, price, item["T"], cp)
        for k, price, cp in zip(market_otm["k"], market_otm["pure_mid"], market_otm["cp_flag"])
    ]
    market_otm["bs_vega"] = [
        bs_vega_forward_unit(k, iv, item["T"])
        for k, iv in zip(market_otm["k"], market_otm["otm_iv"])
    ]
    market_otm = market_otm[
        np.isfinite(market_otm["otm_iv"])
        & (market_otm["otm_iv"] > 0)
        & (market_otm["otm_iv"] < MAX_DISPLAY_IV)
        & (market_otm["bs_vega"] > VEGA_TOL)
    ].sort_values("k")

    fit_otm_prices = np.array([otm_price_from_pure_call(k, c) for k, c in zip(surface_k_grid, pure_fit)])
    fit_cp_flags = np.array([otm_cp_flag_from_k(k) for k in surface_k_grid])
    fit_iv = np.array([
        implied_vol_from_pure_option(k, price, item["T"], cp)
        for k, price, cp in zip(surface_k_grid, fit_otm_prices, fit_cp_flags)
    ])
    fit_vega = np.array([
        bs_vega_forward_unit(k, iv, item["T"])
        for k, iv in zip(surface_k_grid, fit_iv)
    ])
    fit_mask = np.isfinite(fit_iv) & (fit_iv > 0) & (fit_iv < MAX_DISPLAY_IV) & (fit_vega > VEGA_TOL)

    fig_otm_iv_slices.add_trace(
        go.Scatter(
            x=market_otm["k"],
            y=market_otm["otm_iv"],
            mode="markers",
            marker=dict(size=4),
            name=f"market OTM IV {exdate.date()} ({item['dte']}D)",
        )
    )
    fig_otm_iv_slices.add_trace(
        go.Scatter(
            x=surface_k_grid[fit_mask],
            y=fit_iv[fit_mask],
            mode="lines",
            name=f"{fit_label_prefix} OTM IV {exdate.date()} ({item['dte']}D)",
        )
    )

if len(fig_price_slices.data):
    fig_price_slices.update_layout(
        title=f"Pure call price slices: market vs fitted SANOS price ({fit_label_prefix}), {PRODUCT_PREFIX} {TRADE_DATE.date()}",
        xaxis_title="normalized strike k",
        yaxis_title="pure call price",
        width=1150,
        height=760,
    )
    fig_price_slices.show()

    fig_otm_iv_slices.update_layout(
        title=f"OTM IV slices: market OTM IV vs fitted OTM IV ({fit_label_prefix}), {PRODUCT_PREFIX} {TRADE_DATE.date()}",
        xaxis_title="normalized strike k",
        yaxis_title="OTM implied volatility",
        width=1150,
        height=760,
    )
    fig_otm_iv_slices.show()

    print(f"Plotted {len(slice_fit_by_expiry)} expiries with {len(fig_price_slices.data)} price traces and {len(fig_otm_iv_slices.data)} OTM IV traces.")
    if not use_joint_slice_fit:
        print(f"Joint LP was not used for these slice plots: {joint_solution['message']}")
else:
    print("No slice plot traces were produced.")

## 第七部分：翼部稳定化 —— wing-weighted curvature/ridge QP

第六部分的联合 LP 在远翼退化：L1 目标 + 翼部近共线的 BS-call 核 → 最优是多面体的一整个面，simplex 选了个孤立顶点，质量压在 1–2 个 anchor 上（`nonzero_q_count` 长端掉到 7），数据一动就跳、重构 IV 在翼部冲破 300%。

这里把求解升级成**严格凸 QP**（Clarabel 内点法，稀疏）：在原 L1 bid/ask 目标上叠加一个**翼部加权的平滑 + 微小均匀 ridge 惩罚** `λ_s‖W^½ L q‖² + λ_r‖q‖²`，其中平滑权重 `W` 在 ATM core 为 0、只在翼部（`|k-1|` 大、price 上 bid/ask 相对带宽大 = 信息少）打开；均匀 ridge `λ_r` 极小，只保证唯一性。效果：

- 严格凸 → 解唯一、对数据 Lipschitz 连续（消掉"换顶点"式跳变）；
- 翼部把孤立尖峰摊成光滑分布（消掉"1–2 个 anchor 扛旗"），core 覆盖率保持不变；
- 中段（ATM）几乎不动，smile 曲率是真信号、不被压平。

原 LP `joint_solution` 完整保留，新解放在 `joint_solution_qp`，下面做改前/改后对比。

In [ ]:
# --- wing-stabilized joint solve (OSQP sparse QP) ---------------------------
import clarabel
from scipy import sparse as sp

# core vs wing split and penalty strengths (tune with the roughness printout below)
WING_CORE_HALFWIDTH = 0.15    # |k-1| <= this is "core"; beyond is wing
QP_LAM_SMOOTH       = 5.0e-2  # WING fused (first-difference) smoothness strength
QP_LAM_RIDGE        = 1.0e-4  # tiny UNIFORM ridge: strict convexity / uniqueness
QP_WING_GAIN        = 8.0     # how fast the wing weight grows with |k-1| beyond core
QP_SPREAD_INFO_CAP  = 30.0    # cap on low-information (wide-band) up-weighting

def first_difference_operator(n):
    """(n-1) x n first-difference matrix L; row i = q_{i+1}-q_i."""
    if n < 2:
        return sp.csr_matrix((0, n))
    rows = np.repeat(np.arange(n - 1), 2)
    cols = np.column_stack([np.arange(n - 1), np.arange(1, n)]).ravel()
    data = np.column_stack([-np.ones(n - 1), np.ones(n - 1)]).ravel()
    return sp.coo_matrix((data, (rows, cols)), shape=(n - 1, n)).tocsr()

def wing_anchor_weights(item, core_halfwidth=WING_CORE_HALFWIDTH,
                        wing_gain=QP_WING_GAIN, info_cap=QP_SPREAD_INFO_CAP):
    """WING-ONLY smoothness weight: ~0 in the ATM core (leave the real, peaked
    smile curvature untouched) and ramping up in the wings where data is sparse
    and the bid/ask band is wide. Core uniqueness comes from a separate tiny
    uniform ridge, not from this weight."""
    anchor_k = np.asarray(item["anchor_k"], float)
    mk = np.asarray(item["market_k"], float)
    bid = np.asarray(item["pure_bid"], float)
    ask = np.asarray(item["pure_ask"], float)
    mid = np.maximum(0.5 * (bid + ask), 1e-8)
    rel = np.clip((ask - bid) / mid, 1e-4, None)               # relative band per quote
    order = np.argsort(mk)
    rel_at_anchor = np.interp(anchor_k, mk[order], rel[order])
    info = 1.0 / np.maximum(rel_at_anchor, 1e-12)
    med = np.median(info) if len(info) else 1.0
    data_factor = np.clip(med / np.maximum(info, 1e-12), 1.0, info_cap)  # wide band -> bigger
    dist = np.maximum(np.abs(anchor_k - 1.0) - core_halfwidth, 0.0)
    wing_factor = wing_gain * dist          # 0 at |k-1|<=core, grows into the wings
    return data_factor * wing_factor

def solve_joint_expiry_qp(joint_inputs, lam_smooth=QP_LAM_SMOOTH, lam_ridge=QP_LAM_RIDGE):
    exdates = list(joint_inputs.keys())
    q_offsets, u_offsets, v_offsets, a_offsets = {}, {}, {}, {}
    cursor = 0
    for exdate in exdates:
        item = joint_inputs[exdate]
        na, nq = len(item["anchor_k"]), len(item["market_k"])
        q_offsets[exdate] = (cursor, cursor + na); cursor += na
        u_offsets[exdate] = (cursor, cursor + nq); cursor += nq
        v_offsets[exdate] = (cursor, cursor + nq); cursor += nq
        a_offsets[exdate] = (cursor, cursor + nq); cursor += nq
    n_vars = cursor

    c = np.zeros(n_vars)                       # linear term: L1 bid/ask violation + tiny mid tie-breaker
    P_rows, P_cols, P_data = [], [], []        # quadratic term (q-blocks only)

    eq_rows, eq_cols, eq_data, b_eq = [], [], [], []
    ub_rows, ub_cols, ub_data, b_ub = [], [], [], []
    eq_r = ub_r = 0
    def add_eq(idx, val, rhs):
        nonlocal eq_r
        eq_rows.extend([eq_r]*len(idx)); eq_cols.extend(idx); eq_data.extend(val); b_eq.append(rhs); eq_r += 1
    def add_ub(idx, val, rhs):
        nonlocal ub_r
        ub_rows.extend([ub_r]*len(idx)); ub_cols.extend(idx); ub_data.extend(val); b_ub.append(rhs); ub_r += 1

    for exdate in exdates:
        item = joint_inputs[exdate]
        G = item["G"]; anchor_k = item["anchor_k"]
        bid = item["pure_bid"]; mid = item["pure_mid"]; ask = item["pure_ask"]
        weights = normalized_spread_weights(bid, ask)
        n_quotes, n_anchor = G.shape
        q0, q1 = q_offsets[exdate]; u0, u1 = u_offsets[exdate]; v0, v1 = v_offsets[exdate]; a0, a1 = a_offsets[exdate]
        q_idx = np.arange(q0, q1)
        c[u0:u1] = weights; c[v0:v1] = weights; c[a0:a1] = MID_TIE_BREAK_EPS * weights

        # density normalization (same as LP): total mass 1, mean = forward (=1)
        add_eq(q_idx.tolist(), np.ones(n_anchor).tolist(), 1.0)
        add_eq(q_idx.tolist(), anchor_k.tolist(), 1.0)
        for l in range(n_quotes):
            add_ub(np.r_[q_idx, u0 + l].tolist(), np.r_[G[l, :], -1.0].tolist(),  ask[l])
            add_ub(np.r_[q_idx, v0 + l].tolist(), np.r_[-G[l, :], -1.0].tolist(), -bid[l])
            add_ub(np.r_[q_idx, a0 + l].tolist(), np.r_[G[l, :], -1.0].tolist(),  mid[l])
            add_ub(np.r_[q_idx, a0 + l].tolist(), np.r_[-G[l, :], -1.0].tolist(), -mid[l])

        # wing-weighted smoothness + ridge on this expiry's q-block
        w = wing_anchor_weights(item)
        L = first_difference_operator(n_anchor)
        w_edge = 0.5 * (w[:-1] + w[1:])
        # wing-only fused smoothness + tiny UNIFORM ridge (uniqueness everywhere)
        Pq = (lam_smooth * (L.T @ sp.diags(w_edge) @ L) + lam_ridge * sp.identity(n_anchor)).tocoo()
        P_rows.extend((Pq.row + q0).tolist())
        P_cols.extend((Pq.col + q0).tolist())
        P_data.extend((2.0 * Pq.data).tolist())   # OSQP uses 1/2 x'Px

    # calendar no-arbitrage between adjacent expiries (identical to LP)
    calendar_constraint_count = 0
    for prev_exdate, next_exdate in zip(exdates[:-1], exdates[1:]):
        prev_item = joint_inputs[prev_exdate]; next_item = joint_inputs[next_exdate]
        calendar_k = thin_calendar_grid(
            np.union1d(prev_item["market_k"], next_item["market_k"]),
            CALENDAR_DENSE_GRID_POINTS,
            GLOBAL_CALENDAR_K_MIN,
            GLOBAL_CALENDAR_K_MAX,
        )
        prev_eval = build_kernel_matrix(prev_item["anchor_k"], calendar_k, prev_item["smooth_variance"])
        next_eval = build_kernel_matrix(next_item["anchor_k"], calendar_k, next_item["smooth_variance"])
        p0, p1 = q_offsets[prev_exdate]; n0, n1 = q_offsets[next_exdate]
        p_idx = np.arange(p0, p1); n_idx = np.arange(n0, n1)
        for l in range(len(calendar_k)):
            add_ub(np.r_[p_idx, n_idx].tolist(), np.r_[prev_eval[l, :], -next_eval[l, :]].tolist(), 0.0)
            calendar_constraint_count += 1

    A_eq = sp.coo_matrix((eq_data, (eq_rows, eq_cols)), shape=(eq_r, n_vars)).tocsc()
    A_ub = sp.coo_matrix((ub_data, (ub_rows, ub_cols)), shape=(ub_r, n_vars)).tocsc()
    P = sp.coo_matrix((P_data, (P_rows, P_cols)), shape=(n_vars, n_vars)).tocsc()

    # Clarabel form: min 1/2 x'Px + c'x  s.t.  A x + s = b,  s in cones.
    #   equalities  -> ZeroCone   (A_eq x = b_eq)
    #   inequalities-> Nonnegative(A_ub x <= b_ub)  and  (-x <= 0  i.e. x >= 0)
    # Interior-point (Clarabel) converges reliably here; OSQP/ADMM stalls on the
    # wing-collinear, large bid/ask block (17k vars, ~1.9M nnz) -> max-iter.
    b_eq = np.asarray(b_eq); b_ub = np.asarray(b_ub)
    A = sp.vstack([A_eq, A_ub, -sp.identity(n_vars, format="csc")], format="csc")
    b = np.concatenate([b_eq, b_ub, np.zeros(n_vars)])
    cones = [clarabel.ZeroConeT(A_eq.shape[0]),
             clarabel.NonnegativeConeT(A_ub.shape[0] + n_vars)]
    settings = clarabel.DefaultSettings()
    settings.verbose = False
    settings.max_iter = 200
    solver = clarabel.DefaultSolver(sp.triu(P, format="csc"), c, A, b, cones, settings)
    res = solver.solve()
    status = str(res.status)
    success = status == "Solved"

    solution = {"success": success, "message": status, "exdates": exdates,
                "q_offsets": q_offsets, "u_offsets": u_offsets, "v_offsets": v_offsets,
                "a_offsets": a_offsets,
                "calendar_constraint_count": calendar_constraint_count,
                "n_vars": n_vars, "n_eq": A_eq.shape[0], "n_ub": A_ub.shape[0],
                "objective": float(res.obj_val)}
    if not success:
        return solution

    x = np.maximum(np.asarray(res.x), 0.0)   # clean tiny negatives from the IP solve
    per_expiry, q_by_expiry, fit_by_expiry = [], {}, {}
    for exdate in exdates:
        item = joint_inputs[exdate]
        q0, q1 = q_offsets[exdate]; u0, u1 = u_offsets[exdate]; v0, v1 = v_offsets[exdate]; a0, a1 = a_offsets[exdate]
        q = x[q0:q1]; u = x[u0:u1]; v = x[v0:v1]; a = x[a0:a1]
        fit = item["G"] @ q
        bid = item["pure_bid"]; ask = item["pure_ask"]; mid = item["pure_mid"]
        weights = normalized_spread_weights(bid, ask)
        q_by_expiry[exdate] = q; fit_by_expiry[exdate] = fit
        per_expiry.append({
            "exdate": exdate, "dte": item["dte"],
            "n_quotes": len(item["market_k"]), "n_anchor": len(item["anchor_k"]),
            "sum_q": q.sum(), "mean_q": np.dot(q, item["anchor_k"]),
            "nonzero_q_count": int((q > 1e-6).sum()),
            "max_outside_bid_ask": max(u.max(initial=0.0), v.max(initial=0.0)),
            "weighted_outside_error": np.sum(weights * (u + v)),
            "weighted_mid_tiebreak_error": MID_TIE_BREAK_EPS * np.sum(weights * a),
            "mid_rmse": np.sqrt(np.mean((fit - mid) ** 2)),
            "bid_ask_coverage": np.mean((fit >= bid - 1e-9) & (fit <= ask + 1e-9)),
        })
    solution.update({"x": x, "q_by_expiry": q_by_expiry, "fit_by_expiry": fit_by_expiry,
                     "per_expiry": pd.DataFrame(per_expiry)})
    return solution

joint_solution_qp = solve_joint_expiry_qp(joint_inputs)
{"success": joint_solution_qp["success"], "message": joint_solution_qp["message"],
 "objective": joint_solution_qp.get("objective"), "n_vars": joint_solution_qp["n_vars"]}

In [ ]:
# --- before/after: q roughness + core/wing coverage (LP vs QP) --------------
def q_roughness(item, q):
    L = first_difference_operator(len(q))
    return float(np.linalg.norm(L @ q))   # raw |L q|: bigger = spikier/less stable

def wing_core_coverage(item, q):
    k = np.asarray(item["market_k"], float)
    fit = item["G"] @ q
    bid = item["pure_bid"]; ask = item["pure_ask"]
    inside = (fit >= bid - 1e-9) & (fit <= ask + 1e-9)
    wing = np.abs(k - 1.0) > WING_CORE_HALFWIDTH
    core = ~wing
    cov = lambda m: float(np.mean(inside[m])) if m.any() else np.nan
    return cov(core), cov(wing), int(wing.sum())

rows = []
if joint_solution["success"] and joint_solution_qp["success"]:
    for exdate in joint_solution["exdates"]:
        item = joint_inputs[exdate]
        q_lp = joint_solution["q_by_expiry"][exdate]
        q_qp = joint_solution_qp["q_by_expiry"][exdate]
        c_lp, w_lp, n_wing = wing_core_coverage(item, q_lp)
        c_qp, w_qp, _ = wing_core_coverage(item, q_qp)
        rows.append({
            "exdate": pd.Timestamp(exdate).date(), "dte": item["dte"], "n_wing_quotes": n_wing,
            "nnz_lp": int((q_lp > 1e-6).sum()), "nnz_qp": int((q_qp > 1e-6).sum()),
            "rough_lp": q_roughness(item, q_lp), "rough_qp": q_roughness(item, q_qp),
            "wing_cov_lp": w_lp, "wing_cov_qp": w_qp,
            "core_cov_lp": c_lp, "core_cov_qp": c_qp,
        })
wing_compare = pd.DataFrame(rows)
if len(wing_compare):
    print("median q-roughness  LP=%.4g  QP=%.4g  (drop %.0f%%)" % (
        wing_compare["rough_lp"].median(), wing_compare["rough_qp"].median(),
        (1 - wing_compare["rough_qp"].median() / wing_compare["rough_lp"].median()) * 100))
    print("median wing coverage  LP=%.3f  QP=%.3f" % (
        wing_compare["wing_cov_lp"].median(), wing_compare["wing_cov_qp"].median()))
    print("median core coverage  LP=%.3f  QP=%.3f" % (
        wing_compare["core_cov_lp"].median(), wing_compare["core_cov_qp"].median()))
wing_compare

In [ ]:
# --- OTM IV slice: LP vs QP on the longest (worst) expiry -------------------
if joint_solution["success"] and joint_solution_qp["success"]:
    worst_exdate = max(joint_inputs.keys(), key=lambda e: joint_inputs[e]["dte"])
    item = joint_inputs[worst_exdate]
    kg = np.linspace(float(item["market_k"].min()), float(item["market_k"].max()), 201)
    K = build_kernel_matrix(market_k=kg, anchor_k=item["anchor_k"], total_variance=item["smooth_variance"])

    def otm_iv_curve(q):
        pure = K @ q
        prices = np.array([otm_price_from_pure_call(k, c) for k, c in zip(kg, pure)])
        flags = np.array([otm_cp_flag_from_k(k) for k in kg])
        iv = np.array([implied_vol_from_pure_option(k, p, item["T"], cp) for k, p, cp in zip(kg, prices, flags)])
        veg = np.array([bs_vega_forward_unit(k, v, item["T"]) for k, v in zip(kg, iv)])
        m = np.isfinite(iv) & (iv > 0) & (iv < 3.0) & (veg > 1e-4)
        return kg[m], iv[m]

    fig_wing = go.Figure()
    mk_otm = option_chain[(option_chain["exdate"] == pd.Timestamp(worst_exdate))
        & (((option_chain["cp_flag"] == "C") & (option_chain["k"] >= 1.0))
           | ((option_chain["cp_flag"] == "P") & (option_chain["k"] < 1.0)))].copy()
    mk_otm["otm_iv"] = [implied_vol_from_pure_option(k, p, item["T"], cp)
                        for k, p, cp in zip(mk_otm["k"], mk_otm["pure_mid"], mk_otm["cp_flag"])]
    mk_otm = mk_otm[np.isfinite(mk_otm["otm_iv"]) & (mk_otm["otm_iv"] > 0) & (mk_otm["otm_iv"] < 3.0)].sort_values("k")
    fig_wing.add_trace(go.Scatter(x=mk_otm["k"], y=mk_otm["otm_iv"], mode="markers",
                                  marker=dict(size=4, color="black"), name="market OTM IV"))
    x_lp, y_lp = otm_iv_curve(joint_solution["q_by_expiry"][worst_exdate])
    x_qp, y_qp = otm_iv_curve(joint_solution_qp["q_by_expiry"][worst_exdate])
    fig_wing.add_trace(go.Scatter(x=x_lp, y=y_lp, mode="lines", name="joint LP (current)"))
    fig_wing.add_trace(go.Scatter(x=x_qp, y=y_qp, mode="lines", name="wing-stabilized QP"))
    fig_wing.update_layout(
        title=f"OTM IV wing stability: LP vs QP, {pd.Timestamp(worst_exdate).date()} ({item['dte']}D)",
        xaxis_title="normalized strike k", yaxis_title="OTM implied vol", width=1100, height=640)
    fig_wing.show()

## 第八部分：§3.3 完整版 —— 前向乘积鞅（全 strike 精确无套利）

论文 §3.3 / Theorem 3.7 的"formally correct"模型：把每个到期独立选边际，换成**前向乘积鞅**
$$Z_j=\prod_{\ell\le j}M_\ell,\qquad M_\ell\ \text{独立、单位均值}\ (\textstyle\sum_i q^i_\ell K^i_\ell=1)$$
每个增量 $M_\ell$ 是该到期 anchors 上的 lognormal 混合，**第 $i$ 个分量有自己的方差 $dV^i_\ell$**（per-strike，带 skew）。于是 $E[Z_j\mid Z_{j-1}]=Z_{j-1}$ —— **时间方向无套利对所有 strike（含深翼、含外推）按构造精确成立，零 calendar 约束**。

在 log 空间它是个**前向卷积**：$\log Z_j$ 的密度 = $\log Z_{j-1}$ 密度 ⊛ 第 $j$ 个增量密度。给定上一期密度，$C_j(K)$ 对 $q_j$ **线性**，所以每期是个小 LP（复用第六部分的 slack/权重），逐到期前向求解。

**关键标定（论文未细说，本节实测确定）**：增量方差 `dV^i_j` 必须是**小的平滑尺度**（`S33_ETA_S·` per-strike 前向方差增量），不能取完整市场方差——否则"跳跃位置混合的方差 + lognormal 方差"**双重计数**，smile 过宽、拟合崩（实测 eta=1.0→cov 0.12，eta=0.10→cov 0.80）。结果放在 `joint_solution_s33`，与 LP `joint_solution`、QP `joint_solution_qp` 三方对比。

In [ ]:
# --- SANOS section 3.3 (Theorem 3.7) forward multiplicative-martingale, per-strike dV ---
from scipy.optimize import brentq
from scipy.ndimage import gaussian_filter1d

S33_ETA_S = 0.10          # per-strike increment-variance scale (small: jumps carry the smile,
                          # dV only smooths -> avoids variance double-counting). eta=1.0 overshoots.
S33_NX = 65536
_xs = np.linspace(-4.0, 3.0, S33_NX); _dx = _xs[1] - _xs[0]; _ex = np.exp(_xs)

def _s33_total_var(k, C):
    """Invert a pure-call price into call-implied total variance (uses bs_call_forward_unit)."""
    k = float(k); C = float(C); lo = max(1.0 - k, 0.0)
    if not np.isfinite(C) or C <= lo + 1e-12 or C >= 1.0 - 1e-12:
        return np.nan
    try:
        return brentq(lambda w: bs_call_forward_unit(k, w) - C, 1e-10, 9.0, maxiter=100)
    except ValueError:
        return np.nan

def _s33_smile_var(item):
    mk = np.asarray(item["market_k"], float); C = np.asarray(item["pure_mid"], float)
    w = np.array([_s33_total_var(k, c) for k, c in zip(mk, C)]); ok = np.isfinite(w) & (w > 0)
    return mk[ok], w[ok]

def _s33_gsmooth(mass, var):
    sd = np.sqrt(max(var, 0.0))
    return mass.copy() if sd < 0.25 * _dx else gaussian_filter1d(mass, sd / _dx, mode="constant", truncate=6.0)

def _s33_calls(mass, strikes):
    """E[(e^x-K)^+] under the mass vector, exact via suffix sums (e^x ascending)."""
    sme = np.concatenate([np.cumsum((mass * _ex)[::-1])[::-1], [0.0]])
    sm = np.concatenate([np.cumsum(mass[::-1])[::-1], [0.0]])
    strikes = np.asarray(strikes, float); idx = np.searchsorted(_ex, strikes, side="right")
    return sme[idx] - strikes * sm[idx]

def _s33_fit_q(A, bid, mid, ask, K):
    nq, na = A.shape; w = normalized_spread_weights(bid, ask); nv = na + 3 * nq
    Z = np.zeros((nq, nq)); I = np.eye(nq)
    c = np.concatenate([np.zeros(na), w, w, MID_TIE_BREAK_EPS * w])
    Aub = np.block([
        [A, -I,  Z,  Z],
        [-A, Z, -I,  Z],
        [A,  Z,  Z, -I],
        [-A, Z,  Z, -I],
    ])
    bub = np.concatenate([ask, -bid, mid, -mid])
    Aeq = np.zeros((2, nv)); Aeq[0, :na] = 1.0; Aeq[1, :na] = K; beq = np.array([1.0, 1.0])
    return linprog(c, A_ub=Aub, b_ub=bub, A_eq=Aeq, b_eq=beq, bounds=[(0, None)] * nv, method="highs")

def solve_s33_forward_product(joint_inputs, eta_s=S33_ETA_S):
    exdates = sorted(joint_inputs.keys(), key=lambda e: joint_inputs[e]["dte"])
    p_prev = np.zeros(S33_NX); p_prev[np.argmin(np.abs(_xs))] = 1.0
    prev_smk = prev_smw = None
    fine_k = np.linspace(0.20, 3.0, 200); Cfine_prev = None; cal_min_global = np.inf
    per, q_by, dens_by = [], {}, {}
    for e in exdates:
        it = joint_inputs[e]; K = np.asarray(it["anchor_k"], float); mk = np.asarray(it["market_k"], float)
        bid = np.asarray(it["pure_bid"], float); ask = np.asarray(it["pure_ask"], float); mid = np.asarray(it["pure_mid"], float)
        smk, smw = _s33_smile_var(it)
        if len(smk) >= 3:
            wn = np.interp(K, smk, smw)
            dV = np.maximum(eta_s * wn, 1e-7) if prev_smk is None else np.maximum(eta_s * (wn - np.interp(K, prev_smk, prev_smw)), 1e-7)
        else:
            dV = np.full(len(K), max(it["smooth_variance"], 1e-7))
        prev_smk, prev_smw = smk, smw

        cache, A = {}, np.empty((len(mk), len(K)))
        for i in range(len(K)):
            v = dV[i]; key = round(v, 12)
            psm = cache.get(key)
            if psm is None: psm = _s33_gsmooth(p_prev, v); cache[key] = psm
            sc = K[i] * np.exp(-0.5 * v); A[:, i] = sc * _s33_calls(psm, mk / sc)

        res = _s33_fit_q(A, bid, mid, ask, K)
        if not res.success:
            return {"success": False, "message": f"LP failed at dte {it['dte']}: {res.message}"}
        q = res.x[:len(K)]; fit = A @ q
        mu = np.log(K) - 0.5 * dV; p_new = np.zeros(S33_NX)
        for i in range(len(K)):
            if q[i] <= 1e-12: continue
            p_new += q[i] * np.interp(_xs - mu[i], _xs, cache[round(dV[i], 12)], left=0.0, right=0.0)
        s = p_new.sum(); p_new = p_new / s if s > 0 else p_new

        Cfine = _s33_calls(p_new, fine_k)
        cmin = np.inf if Cfine_prev is None else float((Cfine - Cfine_prev).min())
        if np.isfinite(cmin): cal_min_global = min(cal_min_global, cmin)
        q_by[e] = q; dens_by[e] = p_new
        per.append({"exdate": e, "dte": it["dte"], "n_quotes": len(mk), "n_anchor": len(K),
                    "dV_med": float(np.median(dV)),
                    "coverage": float(np.mean((fit >= bid - 1e-9) & (fit <= ask + 1e-9))),
                    "mid_rmse": float(np.sqrt(np.mean((fit - mid) ** 2))),
                    "mean_Z": float(np.dot(p_new, _ex)), "calendar_min_allK": cmin})
        p_prev, Cfine_prev = p_new, Cfine
    return {"success": True, "message": "ok", "exdates": exdates, "eta_s": eta_s,
            "q_by_expiry": q_by, "density_by_expiry": dens_by, "fine_k": fine_k,
            "calendar_min_global": cal_min_global, "per_expiry": pd.DataFrame(per)}

joint_solution_s33 = solve_s33_forward_product(joint_inputs)
{"success": joint_solution_s33["success"], "n_expiries": len(joint_solution_s33.get("exdates", [])),
 "median_coverage": float(joint_solution_s33["per_expiry"]["coverage"].median()) if joint_solution_s33["success"] else None,
 "worst_calendar_diff_allK": joint_solution_s33.get("calendar_min_global")}

In [ ]:
# --- three-way comparison: calendar guarantee over ALL strikes, and coverage ---
def calendar_min_over_grid(sol_q, grid):
    """For an LP/QP solution (mixture weights per expiry), min over the grid of
    C_next - C_prev across adjacent expiries -> all-strike calendar slack."""
    exs = sorted(sol_q.keys(), key=lambda e: joint_inputs[e]["dte"])
    worst = np.inf
    for a, b in zip(exs[:-1], exs[1:]):
        ia, ib = joint_inputs[a], joint_inputs[b]
        Ca = build_kernel_matrix(ia["anchor_k"], grid, ia["smooth_variance"]) @ sol_q[a]
        Cb = build_kernel_matrix(ib["anchor_k"], grid, ib["smooth_variance"]) @ sol_q[b]
        worst = min(worst, float((Cb - Ca).min()))
    return worst

if joint_solution["success"] and joint_solution_qp["success"] and joint_solution_s33["success"]:
    fk = joint_solution_s33["fine_k"]   # 0.20 .. 3.0, incl deep wing & extrapolation
    cal_lp = calendar_min_over_grid(joint_solution["q_by_expiry"], fk)
    cal_qp = calendar_min_over_grid(joint_solution_qp["q_by_expiry"], fk)
    cal_s33 = joint_solution_s33["calendar_min_global"]

    def med_cov(df): return float(df["bid_ask_coverage"].median()) if "bid_ask_coverage" in df else np.nan
    cov_lp = med_cov(joint_solution["per_expiry"])
    cov_qp = med_cov(joint_solution_qp["per_expiry"])
    cov_s33 = float(joint_solution_s33["per_expiry"]["coverage"].median())

    print("                       LP (HiGHS)   QP (Clarabel)   S3.3 (fwd-product)")
    print("calendar constraints   explicit      explicit        NONE (by construction)")
    print("worst calendar diff    %+.2e    %+.2e     %+.2e   <- over ALL strikes [0.2,3.0]" % (cal_lp, cal_qp, cal_s33))
    print("median bid/ask cover   %.3f         %.3f           %.3f" % (cov_lp, cov_qp, cov_s33))
    print("\nNote: LP/QP enforce calendar only on the market-strike grid; S3.3 holds it")
    print("exactly everywhere incl deep wing & extrapolation, with zero calendar constraints.")

joint_s33_per_expiry = joint_solution_s33["per_expiry"] if joint_solution_s33["success"] else pd.DataFrame()
joint_s33_per_expiry

In [ ]:
# --- all-strike calendar slack: S3.3 vs LP, on the longest adjacent pair ---
if joint_solution["success"] and joint_solution_s33["success"]:
    exs = joint_solution_s33["exdates"]; a, b = exs[-2], exs[-1]
    fk = joint_solution_s33["fine_k"]; ia, ib = joint_inputs[a], joint_inputs[b]
    Ca_lp = build_kernel_matrix(ia["anchor_k"], fk, ia["smooth_variance"]) @ joint_solution["q_by_expiry"][a]
    Cb_lp = build_kernel_matrix(ib["anchor_k"], fk, ib["smooth_variance"]) @ joint_solution["q_by_expiry"][b]
    Ca_s = _s33_calls(joint_solution_s33["density_by_expiry"][a], fk)
    Cb_s = _s33_calls(joint_solution_s33["density_by_expiry"][b], fk)
    mk_max = max(ia["market_k"].max(), ib["market_k"].max())
    mk_min = min(ia["market_k"].min(), ib["market_k"].min())

    fig_cal = go.Figure()
    fig_cal.add_trace(go.Scatter(x=fk, y=Cb_lp - Ca_lp, mode="lines", name="LP  C_next - C_prev"))
    fig_cal.add_trace(go.Scatter(x=fk, y=Cb_s - Ca_s, mode="lines", name="S3.3  C_next - C_prev"))
    fig_cal.add_hline(y=0.0, line_dash="dash", line_color="gray")
    fig_cal.add_vrect(x0=mk_min, x1=mk_max, fillcolor="LightGreen", opacity=0.15, line_width=0,
                      annotation_text="market strike range", annotation_position="top left")
    fig_cal.update_layout(
        title=f"All-strike calendar slack (>=0 = arb-free): {pd.Timestamp(a).date()} -> {pd.Timestamp(b).date()} "
              f"({ia['dte']}->{ib['dte']}D). Outside the green band = extrapolation.",
        xaxis_title="normalized strike k", yaxis_title="C_next - C_prev (pure call)", width=1100, height=560)
    fig_cal.show()